In [ ]:

# ============================================================
# OJAS | SIH 2026
# STAGE 1 — DATA INGESTION
#
# Purpose:
# Load the collected ocean data and establish a reproducible
# starting point for LUT generation.
#
# We use pandas for tabular data management and NumPy for
# numerical checks. No environmental values are invented here.
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path

# If you upload the CSV directly to Google Colab, this path
# will normally be /content/ocean_data.csv.
# If your file has a different name, change this variable.

DATA_PATH = "/content/ocean_data.csv"

# Load the dataset.
ocean_df = pd.read_csv(DATA_PATH)

# Preserve the original dataset so later processing does not
# accidentally overwrite the source values.
ocean_raw = ocean_df.copy(deep=True)

print("OJAS ocean dataset loaded successfully.")
print(f"Number of records : {len(ocean_df)}")
print(f"Number of columns : {len(ocean_df.columns)}")
print("\nColumn names:")
print(ocean_df.columns.tolist())

print("\nFirst five records:")
display(ocean_df.head())

OJAS ocean dataset loaded successfully.
Number of records : 1839
Number of columns : 9

Column names:
['Unnamed: 0', 'depth_m', 'temperature_C', 'salinity_PSU', 'turbidity_NTU', 'sound_speed_mps_mackenzie', 'sound_speed_data_mps', 'sound_speed_delta_mps', 'profile_depth_max_m']

First five records:


,Unnamed: 0,depth_m,temperature_C,salinity_PSU,turbidity_NTU,sound_speed_mps_mackenzie,sound_speed_data_mps,sound_speed_delta_mps,profile_depth_max_m
0,0,1.0,14.41920,24.17840,0.5541,1491.957290,1473.875,18.082290,21.0
1,1,2.0,14.51465,34.82145,0.5238,1504.968400,1473.875,31.093400,21.0
2,2,5.0,14.26420,35.93365,0.7444,1505.540125,1473.875,31.665125,21.0
3,3,10.0,14.11650,35.96055,0.8524,1505.178267,1473.875,31.303267,21.0
4,4,20.0,14.07870,35.98490,1.2605,1505.248443,1473.875,31.373443,21.0


In [ ]:
# ============================================================
# OJAS | SIH 2026
# STAGE 2 — ENVIRONMENTAL STATE CONSTRUCTION
# CELL 3 — SINGLE DATASET + DATA PROVENANCE
# ============================================================
#
# IMPORTANT:
# ocean_df is the ONLY dataset used throughout this notebook.
#
# We do not create a second environmental dataset.
# Derived/modelled quantities will simply be added as new
# columns to ocean_df.
#
# This keeps the complete processing chain traceable:
#
# Uploaded ocean data
#        ↓
# ocean_df
#        ↓
# Derived physics quantities
#        ↓
# Environmental states
#        ↓
# Waveform candidates
#        ↓
# Final LUT
# ============================================================

# Make sure the uploaded dataset is loaded.
# If Cell 1 from the previous step was already executed,
# this line simply reloads the same source file.

DATA_PATH = "/content/ocean_data.csv"

ocean_df = pd.read_csv(DATA_PATH)

# Remove an automatically generated CSV index column if present.
index_like_columns = [
    col for col in ocean_df.columns
    if col.lower().startswith("unnamed:")
]

if index_like_columns:
    ocean_df.drop(columns=index_like_columns, inplace=True)

print("OJAS ocean dataset loaded.")
print(f"Rows    : {len(ocean_df)}")
print(f"Columns : {len(ocean_df.columns)}")

print("\nAvailable columns:")
print(ocean_df.columns.tolist())

display(ocean_df.head())

OJAS ocean dataset loaded.
Rows    : 1839
Columns : 8

Available columns:
['depth_m', 'temperature_C', 'salinity_PSU', 'turbidity_NTU', 'sound_speed_mps_mackenzie', 'sound_speed_data_mps', 'sound_speed_delta_mps', 'profile_depth_max_m']


,depth_m,temperature_C,salinity_PSU,turbidity_NTU,sound_speed_mps_mackenzie,sound_speed_data_mps,sound_speed_delta_mps,profile_depth_max_m
0,1.0,14.41920,24.17840,0.5541,1491.957290,1473.875,18.082290,21.0
1,2.0,14.51465,34.82145,0.5238,1504.968400,1473.875,31.093400,21.0
2,5.0,14.26420,35.93365,0.7444,1505.540125,1473.875,31.665125,21.0
3,10.0,14.11650,35.96055,0.8524,1505.178267,1473.875,31.303267,21.0
4,20.0,14.07870,35.98490,1.2605,1505.248443,1473.875,31.373443,21.0


In [ ]:
# ============================================================
# CELL 4 — DATA PROVENANCE AND ENGINEERING ASSUMPTIONS
# ============================================================
#
# We explicitly distinguish:
#
# 1. SOURCE DATA
#    Values obtained from the uploaded ocean dataset.
#
# 2. MODEL OUTPUT
#    Values calculated from equations/models.
#
# 3. REFERENCE
#    Independent/reference values used for comparison.
#
# 4. ASSUMPTION
#    Engineering choices required because hardware or
#    calibration data is not yet available.
# ============================================================

provenance = pd.DataFrame([
    {
        "quantity": "Depth",
        "column": "depth_m",
        "classification": "SOURCE DATA",
        "role": "Ocean/environmental state"
    },
    {
        "quantity": "Temperature",
        "column": "temperature_C",
        "classification": "SOURCE DATA",
        "role": "Mackenzie sound-speed input"
    },
    {
        "quantity": "Salinity",
        "column": "salinity_PSU",
        "classification": "SOURCE DATA",
        "role": "Mackenzie sound-speed input"
    },
    {
        "quantity": "Turbidity",
        "column": "turbidity_NTU",
        "classification": "SOURCE DATA",
        "role": "Scattering-state descriptor"
    },
    {
        "quantity": "Reference sound speed",
        "column": "sound_speed_data_mps",
        "classification": "REFERENCE",
        "role": "Cross-check for Mackenzie model"
    },
    {
        "quantity": "Mackenzie sound speed",
        "column": "sound_speed_mps_mackenzie",
        "classification": "MODEL OUTPUT",
        "role": "Acoustic medium / resolution calculation"
    },
    {
        "quantity": "Sound-speed difference",
        "column": "sound_speed_delta_mps",
        "classification": "DERIVED / COMPARISON",
        "role": "Model-reference consistency check"
    }
])

display(provenance)

# ------------------------------------------------------------
# OJAS engineering decisions
# ------------------------------------------------------------

REQUIRED_RANGE_RESOLUTION_M = 1.0

FC_MIN_HZ = 50_000
FC_MAX_HZ = 600_000

WAVEFORM_FAMILIES = [
    "LFM",
    "Geometric_NFM",
    "Phase_Coded"
]

ENGINEERING_ASSUMPTIONS = {
    "range_resolution_m": REQUIRED_RANGE_RESOLUTION_M,
    "center_frequency_min_kHz": FC_MIN_HZ / 1000,
    "center_frequency_max_kHz": FC_MAX_HZ / 1000,

    # Both operating modes are required:
    # environmental selection and range/mission-aware selection.
    "runtime_mode": "Environmental + Range/Mission",

    "waveform_families": WAVEFORM_FAMILIES,

    # These have NOT been measured/validated yet.
    "transducer_response": "UNVALIDATED",
    "ambient_noise_spectrum": "NOT PROVIDED",
    "bellhop_propagation": "NOT YET VALIDATED",
    "NTU_to_acoustic_loss": "NOT CALIBRATED",

    # Candidate selection will use multiple metrics.
    "optimization": "Balanced multi-metric selection"
}

print("\nENGINEERING ASSUMPTIONS / DESIGN DECISIONS")

for key, value in ENGINEERING_ASSUMPTIONS.items():
    print(f"{key}: {value}")

,quantity,column,classification,role
0,Depth,depth_m,SOURCE DATA,Ocean/environmental state
1,Temperature,temperature_C,SOURCE DATA,Mackenzie sound-speed input
2,Salinity,salinity_PSU,SOURCE DATA,Mackenzie sound-speed input
3,Turbidity,turbidity_NTU,SOURCE DATA,Scattering-state descriptor
4,Reference sound speed,sound_speed_data_mps,REFERENCE,Cross-check for Mackenzie model
5,Mackenzie sound speed,sound_speed_mps_mackenzie,MODEL OUTPUT,Acoustic medium / resolution calculation
6,Sound-speed difference,sound_speed_delta_mps,DERIVED / COMPARISON,Model-reference consistency check



ENGINEERING ASSUMPTIONS / DESIGN DECISIONS
range_resolution_m: 1.0
center_frequency_min_kHz: 50.0
center_frequency_max_kHz: 600.0
runtime_mode: Environmental + Range/Mission
waveform_families: ['LFM', 'Geometric_NFM', 'Phase_Coded']
transducer_response: UNVALIDATED
ambient_noise_spectrum: NOT PROVIDED
bellhop_propagation: NOT YET VALIDATED
NTU_to_acoustic_loss: NOT CALIBRATED
optimization: Balanced multi-metric selection


In [ ]:
# ============================================================
# CELL 5 — MACKENZIE CROSS-CHECK + RESOLUTION REQUIREMENT
# ============================================================
#
# Methodology:
#
#        ΔR ≈ c / (2B)
#
# Therefore:
#
#        B_min = c / (2 ΔR_max)
#
# For our requirement:
#
#        ΔR_max = 1 metre
#
# B_min is a DERIVED PHYSICS VALUE.
# It is NOT a final LUT bandwidth.
#
# ============================================================

# ------------------------------------------------------------
# 1. Copy supplied values into clearly named analysis columns
# ------------------------------------------------------------

ocean_df["c_mackenzie_mps"] = (
    ocean_df["sound_speed_mps_mackenzie"]
)

ocean_df["c_reference_mps"] = (
    ocean_df["sound_speed_data_mps"]
)

# ------------------------------------------------------------
# 2. Model-reference comparison
# ------------------------------------------------------------

ocean_df["c_error_mps"] = (
    ocean_df["c_mackenzie_mps"]
    - ocean_df["c_reference_mps"]
)

ocean_df["c_abs_error_mps"] = (
    ocean_df["c_error_mps"].abs()
)

ocean_df["c_percent_difference"] = (
    ocean_df["c_error_mps"]
    / ocean_df["c_reference_mps"]
) * 100

# ------------------------------------------------------------
# 3. Minimum bandwidth for 1 m range resolution
# ------------------------------------------------------------

ocean_df["B_min_Hz"] = (
    ocean_df["c_mackenzie_mps"]
    / (2 * REQUIRED_RANGE_RESOLUTION_M)
)

ocean_df["B_min_kHz"] = (
    ocean_df["B_min_Hz"] / 1000
)

# Verify the resulting theoretical resolution.
ocean_df["resolution_from_Bmin_m"] = (
    ocean_df["c_mackenzie_mps"]
    / (2 * ocean_df["B_min_Hz"])
)

# ------------------------------------------------------------
# 4. Display results
# ------------------------------------------------------------

print("MACKENZIE vs REFERENCE SOUND SPEED")
display(
    ocean_df[
        [
            "depth_m",
            "temperature_C",
            "salinity_PSU",
            "c_mackenzie_mps",
            "c_reference_mps",
            "c_error_mps",
            "c_percent_difference"
        ]
    ].head(10)
)

print("\nMINIMUM BANDWIDTH REQUIRED FOR 1 m RESOLUTION")

print(
    f"Minimum B_min : "
    f"{ocean_df['B_min_kHz'].min():.3f} kHz"
)

print(
    f"Maximum B_min : "
    f"{ocean_df['B_min_kHz'].max():.3f} kHz"
)

print(
    f"Mean B_min    : "
    f"{ocean_df['B_min_kHz'].mean():.3f} kHz"
)

print(
    f"Median B_min  : "
    f"{ocean_df['B_min_kHz'].median():.3f} kHz"
)

print("\nResolution verification:")
print(
    "Maximum absolute error from requested 1 m =",
    np.max(
        np.abs(
            ocean_df["resolution_from_Bmin_m"]
            - REQUIRED_RANGE_RESOLUTION_M
        )
    ),
    "m"
)

print("\nCurrent ocean_df now contains both:")
print("  • original uploaded/source values")
print("  • clearly labelled derived/model quantities")

MACKENZIE vs REFERENCE SOUND SPEED


,depth_m,temperature_C,salinity_PSU,c_mackenzie_mps,c_reference_mps,c_error_mps,c_percent_difference
0,1.0,14.41920,24.17840,1491.957290,1473.875,18.082290,1.226854
1,2.0,14.51465,34.82145,1504.968400,1473.875,31.093400,2.109636
2,5.0,14.26420,35.93365,1505.540125,1473.875,31.665125,2.148427
3,10.0,14.11650,35.96055,1505.178267,1473.875,31.303267,2.123875
4,20.0,14.07870,35.98490,1505.248443,1473.875,31.373443,2.128637
5,1.0,16.83100,25.05710,1500.745861,1432.255,68.490861,4.782030
6,2.0,16.64970,34.94135,1511.755161,1432.255,79.500161,5.550699
7,5.0,15.75525,36.10045,1510.433185,1432.255,78.178185,5.458398
8,10.0,15.37495,36.06810,1509.297087,1432.255,77.042087,5.379076
9,20.0,15.15875,36.05090,1508.763803,1432.255,76.508803,5.341842



MINIMUM BANDWIDTH REQUIRED FOR 1 m RESOLUTION
Minimum B_min : 0.737 kHz
Maximum B_min : 0.779 kHz
Mean B_min    : 0.759 kHz
Median B_min  : 0.761 kHz

Resolution verification:
Maximum absolute error from requested 1 m = 0.0 m

Current ocean_df now contains both:
  • original uploaded/source values
  • clearly labelled derived/model quantities


In [ ]:
# ============================================================
# STAGE 3 — WAVEFORM CANDIDATE GENERATION
# Cell 1: Engineering parameters and candidate design space
# ============================================================

import numpy as np
import pandas as pd

# -----------------------------
# Hardware / system assumptions
# -----------------------------
FS_HZ = 2_000_000                 # 2 MHz sampling rate
NYQUIST_HZ = FS_HZ / 2

FC_MIN_HZ = 50_000
FC_MAX_HZ = 600_000

# From Stage 2
B_MIN_HZ = float(ocean_df["B_min_Hz"].max())

# Small safety margin above the absolute minimum bandwidth.
# This avoids designing exactly on the theoretical resolution boundary.
B_MARGIN = 1.10
B_BASE_HZ = B_MIN_HZ * B_MARGIN

# Pulse-duration range for candidate generation.
# These are design-space values, NOT measured hardware limits.
PULSE_DURATIONS_US = np.array([
    100, 200, 500, 1000, 2000
])

# Center-frequency design points.
# We deliberately cover the complete allowed 50–600 kHz range.
CENTER_FREQUENCIES_KHZ = np.array([
    50, 75, 100, 150, 200, 300, 400, 500, 600
])

# Bandwidth multipliers relative to the minimum required bandwidth.
# The final optimizer will decide which are actually useful.
BANDWIDTH_MULTIPLIERS = np.array([
    1.0, 2.0, 4.0, 8.0, 16.0
])

WAVEFORM_FAMILIES = [
    "LFM",
    "Geometric_NFM",
    "Phase_Coded"
]

print("STAGE 3 DESIGN SPACE")
print("-" * 50)
print(f"Sampling frequency : {FS_HZ/1e6:.2f} MHz")
print(f"Nyquist frequency   : {NYQUIST_HZ/1e3:.1f} kHz")
print(f"Center frequency   : {FC_MIN_HZ/1e3:.0f}–{FC_MAX_HZ/1e3:.0f} kHz")
print(f"Stage-2 B_min max  : {B_MIN_HZ/1e3:.3f} kHz")
print(f"Design B_base      : {B_BASE_HZ/1e3:.3f} kHz")
print(f"Pulse durations    : {PULSE_DURATIONS_US} µs")
print(f"Waveform families  : {WAVEFORM_FAMILIES}")

STAGE 3 DESIGN SPACE
--------------------------------------------------
Sampling frequency : 2.00 MHz
Nyquist frequency   : 1000.0 kHz
Center frequency   : 50–600 kHz
Stage-2 B_min max  : 0.779 kHz
Design B_base      : 0.857 kHz
Pulse durations    : [ 100  200  500 1000 2000] µs
Waveform families  : ['LFM', 'Geometric_NFM', 'Phase_Coded']


In [ ]:
# ============================================================
# STAGE 3 — Cell 2
# Generate candidate waveform parameter combinations
# ============================================================

candidate_rows = []

candidate_id = 1

for waveform_type in WAVEFORM_FAMILIES:

    for fc_khz in CENTER_FREQUENCIES_KHZ:

        fc_hz = fc_khz * 1000

        for bw_mult in BANDWIDTH_MULTIPLIERS:

            B_hz = B_BASE_HZ * bw_mult

            for Tp_us in PULSE_DURATIONS_US:

                Tp_s = Tp_us * 1e-6

                f_low = fc_hz - B_hz / 2
                f_high = fc_hz + B_hz / 2

                # Nyquist feasibility
                nyquist_valid = f_high <= NYQUIST_HZ

                # Frequency-range feasibility
                frequency_valid = (
                    f_low >= FC_MIN_HZ and
                    f_high <= FC_MAX_HZ
                )

                # Theoretical range resolution
                # c will be evaluated later for each ocean state.
                # Here we retain B as the waveform design parameter.
                TBP = B_hz * Tp_s

                candidate_rows.append({
                    "candidate_id": candidate_id,
                    "waveform_type": waveform_type,
                    "fc_Hz": fc_hz,
                    "fc_kHz": fc_khz,
                    "B_Hz": B_hz,
                    "B_kHz": B_hz / 1000,
                    "Tp_us": Tp_us,
                    "Tp_s": Tp_s,
                    "TBP": TBP,
                    "f_low_Hz": f_low,
                    "f_high_Hz": f_high,
                    "nyquist_valid": nyquist_valid,
                    "frequency_valid": frequency_valid
                })

                candidate_id += 1

candidate_df = pd.DataFrame(candidate_rows)

print("Candidate pool created.")
print(f"Total candidates : {len(candidate_df)}")

print("\nCandidates by waveform family:")
print(candidate_df["waveform_type"].value_counts())

print("\nFrequency range:")
print(
    f"{candidate_df['fc_kHz'].min():.0f}–"
    f"{candidate_df['fc_kHz'].max():.0f} kHz"
)

print("\nBandwidth range:")
print(
    f"{candidate_df['B_kHz'].min():.3f}–"
    f"{candidate_df['B_kHz'].max():.3f} kHz"
)

print("\nNyquist-invalid candidates:",
      (~candidate_df["nyquist_valid"]).sum())

print("\nFrequency-invalid candidates:",
      (~candidate_df["frequency_valid"]).sum())

candidate_df.head()

Candidate pool created.
Total candidates : 675

Candidates by waveform family:
waveform_type
LFM              225
Geometric_NFM    225
Phase_Coded      225
Name: count, dtype: int64

Frequency range:
50–600 kHz

Bandwidth range:
0.857–13.718 kHz

Nyquist-invalid candidates: 0

Frequency-invalid candidates: 150


,candidate_id,waveform_type,fc_Hz,fc_kHz,B_Hz,B_kHz,Tp_us,Tp_s,TBP,f_low_Hz,f_high_Hz,nyquist_valid,frequency_valid
0,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,50428.672199,True,False
1,2,LFM,50000,50,857.344397,0.857344,200,0.0002,0.171469,49571.327801,50428.672199,True,False
2,3,LFM,50000,50,857.344397,0.857344,500,0.0005,0.428672,49571.327801,50428.672199,True,False
3,4,LFM,50000,50,857.344397,0.857344,1000,0.0010,0.857344,49571.327801,50428.672199,True,False
4,5,LFM,50000,50,857.344397,0.857344,2000,0.0020,1.714689,49571.327801,50428.672199,True,False


In [ ]:
# ============================================================
# STAGE 3 — Cell 3
# Validate the candidate design space
# ============================================================

# Hard hardware feasibility for this stage
candidate_df["hardware_frequency_valid"] = (
    candidate_df["frequency_valid"]
    & candidate_df["nyquist_valid"]
)

# Sampling-rate requirement from methodology:
# fs >= 2 * (fc + B/2)
candidate_df["fs_required_Hz"] = (
    2 * candidate_df["f_high_Hz"]
)

candidate_df["fs_margin_Hz"] = (
    FS_HZ - candidate_df["fs_required_Hz"]
)

candidate_df["sampling_valid"] = (
    candidate_df["fs_required_Hz"] <= FS_HZ
)

print("STAGE 3 INITIAL VALIDATION")
print("-" * 50)

print(
    "Sampling-valid candidates:",
    candidate_df["sampling_valid"].sum(),
    "/",
    len(candidate_df)
)

print(
    "Frequency-valid candidates:",
    candidate_df["frequency_valid"].sum(),
    "/",
    len(candidate_df)
)

print(
    "Hardware-frequency-valid candidates:",
    candidate_df["hardware_frequency_valid"].sum(),
    "/",
    len(candidate_df)
)

print(
    "\nMinimum sampling margin:",
    f"{candidate_df['fs_margin_Hz'].min()/1000:.2f} kHz"
)

print(
    "Maximum required sampling rate:",
    f"{candidate_df['fs_required_Hz'].max()/1e6:.4f} MHz"
)

STAGE 3 INITIAL VALIDATION
--------------------------------------------------
Sampling-valid candidates: 675 / 675
Frequency-valid candidates: 525 / 675
Hardware-frequency-valid candidates: 525 / 675

Minimum sampling margin: 786.28 kHz
Maximum required sampling rate: 1.2137 MHz


In [ ]:
# ============================================================
# STAGE 3 — Cell 4
# Candidate-level physical waveform metrics
# ============================================================

# Use the environmental Mackenzie sound-speed distribution
# to evaluate the resolution capability of each candidate.

C_MIN_MPS = ocean_df["c_mackenzie_mps"].min()
C_MEAN_MPS = ocean_df["c_mackenzie_mps"].mean()
C_MAX_MPS = ocean_df["c_mackenzie_mps"].max()

# Conservative resolution calculation:
# higher sound speed gives the worst-case (largest) resolution
candidate_df["resolution_worst_case_m"] = (
    C_MAX_MPS / (2 * candidate_df["B_Hz"])
)

# Typical/mean environmental resolution
candidate_df["resolution_mean_environment_m"] = (
    C_MEAN_MPS / (2 * candidate_df["B_Hz"])
)

# Best-case resolution
candidate_df["resolution_best_case_m"] = (
    C_MIN_MPS / (2 * candidate_df["B_Hz"])
)

# Check whether the waveform satisfies the requested 1 m resolution
# throughout the available environmental sound-speed range.
candidate_df["resolution_1m_valid"] = (
    candidate_df["resolution_worst_case_m"] <= REQUIRED_RANGE_RESOLUTION_M
)

# TBP
candidate_df["TBP"] = (
    candidate_df["B_Hz"] * candidate_df["Tp_s"]
)

# Occupied bandwidth
candidate_df["occupied_bandwidth_Hz"] = (
    candidate_df["f_high_Hz"] - candidate_df["f_low_Hz"]
)

print("CANDIDATE PHYSICS METRICS")
print("-" * 50)

print(f"Mackenzie c minimum : {C_MIN_MPS:.2f} m/s")
print(f"Mackenzie c mean    : {C_MEAN_MPS:.2f} m/s")
print(f"Mackenzie c maximum : {C_MAX_MPS:.2f} m/s")

print(
    "\nCandidates satisfying 1 m resolution everywhere:",
    candidate_df["resolution_1m_valid"].sum(),
    "/",
    len(candidate_df)
)

print(
    "Candidates requiring >1 m resolution somewhere:",
    (~candidate_df["resolution_1m_valid"]).sum()
)

print("\nResolution range across candidate pool:")
print(
    f"Best possible     : "
    f"{candidate_df['resolution_best_case_m'].min():.3f} m"
)
print(
    f"Worst possible    : "
    f"{candidate_df['resolution_worst_case_m'].max():.3f} m"
)

candidate_df[
    [
        "candidate_id",
        "waveform_type",
        "fc_kHz",
        "B_kHz",
        "Tp_us",
        "TBP",
        "resolution_worst_case_m",
        "resolution_1m_valid"
    ]
].head(10)

CANDIDATE PHYSICS METRICS
--------------------------------------------------
Mackenzie c minimum : 1473.99 m/s
Mackenzie c mean    : 1518.34 m/s
Mackenzie c maximum : 1558.81 m/s

Candidates satisfying 1 m resolution everywhere: 675 / 675
Candidates requiring >1 m resolution somewhere: 0

Resolution range across candidate pool:
Best possible     : 0.054 m
Worst possible    : 0.909 m


,candidate_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,resolution_worst_case_m,resolution_1m_valid
0,1,LFM,50,0.857344,100,0.085734,0.909091,True
1,2,LFM,50,0.857344,200,0.171469,0.909091,True
2,3,LFM,50,0.857344,500,0.428672,0.909091,True
3,4,LFM,50,0.857344,1000,0.857344,0.909091,True
4,5,LFM,50,0.857344,2000,1.714689,0.909091,True
5,6,LFM,50,1.714689,100,0.171469,0.454545,True
6,7,LFM,50,1.714689,200,0.342938,0.454545,True
7,8,LFM,50,1.714689,500,0.857344,0.454545,True
8,9,LFM,50,1.714689,1000,1.714689,0.454545,True
9,10,LFM,50,1.714689,2000,3.429378,0.454545,True


In [ ]:
# ============================================================
# STAGE 3 — Cell 5
# Candidate-space engineering summary
# ============================================================

summary_by_family = (
    candidate_df
    .groupby("waveform_type")
    .agg(
        candidates=("candidate_id", "count"),
        resolution_valid=("resolution_1m_valid", "sum"),
        fc_min_kHz=("fc_kHz", "min"),
        fc_max_kHz=("fc_kHz", "max"),
        B_min_kHz=("B_kHz", "min"),
        B_max_kHz=("B_kHz", "max"),
        TBP_min=("TBP", "min"),
        TBP_max=("TBP", "max"),
        resolution_best_m=("resolution_best_case_m", "min"),
        resolution_worst_m=("resolution_worst_case_m", "max")
    )
    .reset_index()
)

print("CANDIDATE SPACE BY WAVEFORM FAMILY")
print("-" * 80)
display(summary_by_family)

print("\nResolution-valid candidates by bandwidth:")
display(
    candidate_df
    .groupby("B_kHz")
    .agg(
        candidates=("candidate_id", "count"),
        resolution_valid=("resolution_1m_valid", "sum"),
        worst_case_resolution_m=("resolution_worst_case_m", "max"),
        mean_TBP=("TBP", "mean")
    )
    .reset_index()
)

CANDIDATE SPACE BY WAVEFORM FAMILY
--------------------------------------------------------------------------------


,waveform_type,candidates,resolution_valid,fc_min_kHz,fc_max_kHz,B_min_kHz,B_max_kHz,TBP_min,TBP_max,resolution_best_m,resolution_worst_m
0,Geometric_NFM,225,225,50,600,0.857344,13.71751,0.085734,27.435021,0.053727,0.909091
1,LFM,225,225,50,600,0.857344,13.71751,0.085734,27.435021,0.053727,0.909091
2,Phase_Coded,225,225,50,600,0.857344,13.71751,0.085734,27.435021,0.053727,0.909091



Resolution-valid candidates by bandwidth:


,B_kHz,candidates,resolution_valid,worst_case_resolution_m,mean_TBP
0,0.857344,135,135,0.909091,0.651582
1,1.714689,135,135,0.454545,1.303163
2,3.429378,135,135,0.227273,2.606327
3,6.858755,135,135,0.113636,5.212654
4,13.717510,135,135,0.056818,10.425308


In [ ]:
# ============================================================
# STAGE 4 — DETECTION / PROPAGATION MODEL
# Cell 1: Unvalidated scenario assumptions
#
# IMPORTANT:
# These values are NOT measured/calibrated values.
# They are engineering placeholders used only to make the
# optimization pipeline executable.
# ============================================================

scenario_df = pd.DataFrame([
    {
        "scenario_id": "S1_Near_LowNoise",
        "range_m": 100,
        "ambient_noise_dB": 50,
        "target_strength_dB": 10,
        "transmit_level_dB": 180,
        "detection_threshold_dB": 10
    },
    {
        "scenario_id": "S2_Medium",
        "range_m": 500,
        "ambient_noise_dB": 65,
        "target_strength_dB": 0,
        "transmit_level_dB": 180,
        "detection_threshold_dB": 10
    },
    {
        "scenario_id": "S3_Far_HighNoise",
        "range_m": 1000,
        "ambient_noise_dB": 80,
        "target_strength_dB": -10,
        "transmit_level_dB": 180,
        "detection_threshold_dB": 10
    }
])

# Explicit provenance labels
scenario_df["noise_status"] = "UNVALIDATED_ASSUMPTION"
scenario_df["target_strength_status"] = "UNVALIDATED_ASSUMPTION"
scenario_df["transmit_level_status"] = "UNVALIDATED_ASSUMPTION"
scenario_df["detection_threshold_status"] = "UNVALIDATED_ASSUMPTION"

print("STAGE 4 — SCENARIO ASSUMPTIONS")
print("-" * 70)
display(scenario_df)

print("\nIMPORTANT:")
print("Noise, target strength, transmit level and detection threshold")
print("are scenario assumptions, NOT experimental measurements.")

STAGE 4 — SCENARIO ASSUMPTIONS
----------------------------------------------------------------------


,scenario_id,range_m,ambient_noise_dB,target_strength_dB,transmit_level_dB,detection_threshold_dB,noise_status,target_strength_status,transmit_level_status,detection_threshold_status
0,S1_Near_LowNoise,100,50,10,180,10,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION
1,S2_Medium,500,65,0,180,10,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION
2,S3_Far_HighNoise,1000,80,-10,180,10,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION,UNVALIDATED_ASSUMPTION



IMPORTANT:
Noise, target strength, transmit level and detection threshold
are scenario assumptions, NOT experimental measurements.


In [ ]:
# ============================================================
# STAGE 4 — Cell 2
# Transparent propagation-loss proxy
# ============================================================

def absorption_proxy_db_per_km(fc_khz):
    """
    Unvalidated frequency-dependent absorption proxy.

    PURPOSE:
        Provide a transparent frequency penalty for the
        first optimization pass.

    STATUS:
        MODEL ASSUMPTION — NOT CALIBRATED.

    NOTE:
        This function is intentionally kept separate so it can
        later be replaced by a validated absorption model.
    """

    # Smooth frequency-dependent proxy.
    # It increases with frequency and is used only for
    # relative candidate comparison.
    return 0.02 + 0.00008 * (fc_khz ** 1.5)


def transmission_loss_proxy_db(fc_khz, range_m):
    """
    Spherical spreading + frequency-dependent absorption.
    """

    range_km = max(range_m / 1000.0, 0.001)

    spreading_loss_db = 20 * np.log10(max(range_m, 1.0))

    alpha_db_per_km = absorption_proxy_db_per_km(fc_khz)

    absorption_loss_db = alpha_db_per_km * range_km

    return spreading_loss_db + absorption_loss_db


print("Propagation proxy created.")
print("Status: UNVALIDATED MODEL")
print()
print("Example transmission losses:")

for fc in [50, 100, 300, 600]:
    tl = transmission_loss_proxy_db(fc, 1000)
    print(f"{fc:>3} kHz @ 1000 m -> TL ≈ {tl:.2f} dB")

Propagation proxy created.
Status: UNVALIDATED MODEL

Example transmission losses:
 50 kHz @ 1000 m -> TL ≈ 60.05 dB
100 kHz @ 1000 m -> TL ≈ 60.10 dB
300 kHz @ 1000 m -> TL ≈ 60.44 dB
600 kHz @ 1000 m -> TL ≈ 61.20 dB


In [ ]:
# ============================================================
# STAGE 4 — Cell 3
# Candidate × scenario detection model
# ============================================================

detection_rows = []

for _, candidate in candidate_df.iterrows():

    fc_khz = candidate["fc_kHz"]
    B_hz = candidate["B_Hz"]
    Tp_s = candidate["Tp_s"]

    # Processing gain
    TBP = max(B_hz * Tp_s, 1.0)
    processing_gain_db = 10 * np.log10(TBP)

    for _, scenario in scenario_df.iterrows():

        range_m = scenario["range_m"]

        # Propagation loss proxy
        TL_db = transmission_loss_proxy_db(
            fc_khz,
            range_m
        )

        # Simplified detection-excess proxy:
        #
        # SE_proxy =
        #     SL + PG + TS - 2TL - NL - DT
        #
        # This follows the structure of the methodology,
        # but the inputs are unvalidated assumptions.

        SE_proxy_db = (
            scenario["transmit_level_dB"]
            + processing_gain_db
            + scenario["target_strength_dB"]
            - 2 * TL_db
            - scenario["ambient_noise_dB"]
            - scenario["detection_threshold_dB"]
        )

        detection_rows.append({
            "candidate_id": candidate["candidate_id"],
            "scenario_id": scenario["scenario_id"],
            "fc_kHz": fc_khz,
            "B_kHz": candidate["B_kHz"],
            "Tp_us": candidate["Tp_us"],
            "waveform_type": candidate["waveform_type"],
            "TBP": candidate["TBP"],
            "processing_gain_dB": processing_gain_db,
            "TL_proxy_dB": TL_db,
            "SE_proxy_dB": SE_proxy_db
        })

detection_df = pd.DataFrame(detection_rows)

print("Detection model generated.")
print("-" * 60)
print(f"Candidates        : {candidate_df['candidate_id'].nunique()}")
print(f"Scenarios         : {scenario_df['scenario_id'].nunique()}")
print(f"Candidate-scenario evaluations : {len(detection_df)}")

print("\nDetection-excess proxy statistics:")
display(
    detection_df["SE_proxy_dB"].describe()
)

print("\nDetection proxy by scenario:")
display(
    detection_df
    .groupby("scenario_id")["SE_proxy_dB"]
    .agg(["min", "mean", "max"])
    .reset_index()
)

Detection model generated.
------------------------------------------------------------
Candidates        : 675
Scenarios         : 3
Candidate-scenario evaluations : 2025

Detection-excess proxy statistics:


,SE_proxy_dB
count,2025.000000
mean,5.589091
std,37.501582
min,-42.391510
25%,-34.744415
50%,-1.531374
75%,49.966606
max,64.373396



Detection proxy by scenario:


,scenario_id,min,mean,max
0,S1_Near_LowNoise,49.760849,53.611741,64.373396
1,S2_Medium,-4.154555,0.311664,11.375968
2,S3_Far_HighNoise,-42.391510,-37.156132,-25.713516


In [ ]:
# ============================================================
# STAGE 4 — Cell 4
# Hard feasibility constraints
# ============================================================

# Required detection excess.
# This is an engineering assumption for the first simulation.
SE_MIN_DB = 10.0

# Maximum normalized pulse-energy budget.
# We will establish the actual normalized value from the
# feasible candidate population rather than pretending to
# know the real transmitter energy.
#
# This parameter is deliberately NOT used yet.
ENERGY_LIMIT_NORMALIZED = 1.0

# Candidate-level hard constraints
candidate_df["resolution_valid"] = (
    candidate_df["resolution_1m_valid"]
)

candidate_df["frequency_valid_final"] = (
    candidate_df["frequency_valid"]
)

candidate_df["sampling_valid_final"] = (
    candidate_df["sampling_valid"]
)

print("HARD CONSTRAINTS")
print("-" * 60)
print(f"Minimum detection excess : {SE_MIN_DB:.1f} dB")
print(f"Required resolution      : {REQUIRED_RANGE_RESOLUTION_M:.1f} m")
print(f"Sampling rate            : {FS_HZ/1e6:.2f} MHz")
print(f"Frequency range          : "
      f"{FC_MIN_HZ/1e3:.0f}–{FC_MAX_HZ/1e3:.0f} kHz")

print("\nCandidate-level feasibility:")
print(
    "Resolution valid :",
    candidate_df["resolution_valid"].sum(),
    "/",
    len(candidate_df)
)

print(
    "Frequency valid  :",
    candidate_df["frequency_valid_final"].sum(),
    "/",
    len(candidate_df)
)

print(
    "Sampling valid   :",
    candidate_df["sampling_valid_final"].sum(),
    "/",
    len(candidate_df)
)

HARD CONSTRAINTS
------------------------------------------------------------
Minimum detection excess : 10.0 dB
Required resolution      : 1.0 m
Sampling rate            : 2.00 MHz
Frequency range          : 50–600 kHz

Candidate-level feasibility:
Resolution valid : 675 / 675
Frequency valid  : 525 / 675
Sampling valid   : 675 / 675


In [ ]:
# ============================================================
# STAGE 4 — Cell 5
# Minimum amplitude and relative energy proxy
# ============================================================

# Merge candidate parameters with detection results
evaluation_df = detection_df.merge(
    candidate_df[
        [
            "candidate_id",
            "resolution_valid",
            "frequency_valid_final",
            "sampling_valid_final"
        ]
    ],
    on="candidate_id",
    how="left"
)

# ------------------------------------------------------------
# Minimum normalized amplitude
# ------------------------------------------------------------
#
# If SE_proxy >= SE_MIN:
#
# amplitude_required = 10^((SE_MIN - SE_proxy)/20)
#
# This means:
#   SE exactly at threshold -> amplitude = 1
#   SE above threshold      -> amplitude < 1
#
# Candidates below threshold are marked infeasible rather than
# allowing the amplitude model to hide the detection failure.

evaluation_df["detection_valid"] = (
    evaluation_df["SE_proxy_dB"] >= SE_MIN_DB
)

evaluation_df["amplitude_required_normalized"] = np.nan

valid_detection = evaluation_df["detection_valid"]

evaluation_df.loc[
    valid_detection,
    "amplitude_required_normalized"
] = (
    10 ** (
        (SE_MIN_DB - evaluation_df.loc[
            valid_detection,
            "SE_proxy_dB"
        ]) / 20
    )
)

# ------------------------------------------------------------
# Relative pulse-energy proxy
# ------------------------------------------------------------
#
# E ∝ A² × Tp
#
# Since absolute transducer voltage/current/efficiency data
# are unavailable, this is ONLY a relative energy metric.

evaluation_df["energy_proxy"] = np.nan

evaluation_df.loc[
    valid_detection,
    "energy_proxy"
] = (
    evaluation_df.loc[
        valid_detection,
        "amplitude_required_normalized"
    ] ** 2
    *
    evaluation_df.loc[
        valid_detection,
        "Tp_us"
    ]
)

# ------------------------------------------------------------
# Overall hard feasibility
# ------------------------------------------------------------

evaluation_df["hard_feasible"] = (
    evaluation_df["detection_valid"]
    & evaluation_df["resolution_valid"]
    & evaluation_df["frequency_valid_final"]
    & evaluation_df["sampling_valid_final"]
)

print("ENERGY / FEASIBILITY MODEL")
print("-" * 60)

print(
    "Detection-feasible evaluations:",
    evaluation_df["detection_valid"].sum(),
    "/",
    len(evaluation_df)
)

print(
    "Hard-feasible evaluations:",
    evaluation_df["hard_feasible"].sum(),
    "/",
    len(evaluation_df)
)

print("\nHard-feasible evaluations by scenario:")
display(
    evaluation_df
    .groupby("scenario_id")["hard_feasible"]
    .agg(["sum", "count"])
    .reset_index()
)

print("\nEnergy proxy statistics for hard-feasible candidates:")
display(
    evaluation_df.loc[
        evaluation_df["hard_feasible"],
        "energy_proxy"
    ].describe()
)

ENERGY / FEASIBILITY MODEL
------------------------------------------------------------
Detection-feasible evaluations: 702 / 2025
Hard-feasible evaluations: 546 / 2025

Hard-feasible evaluations by scenario:


,scenario_id,sum,count
0,S1_Near_LowNoise,525,675
1,S2_Medium,21,675
2,S3_Far_HighNoise,0,675



Energy proxy statistics for hard-feasible candidates:


,energy_proxy
count,546.000000
mean,60.522392
std,303.496986
min,0.007314
25%,0.010077
50%,0.017637
75%,0.050231
max,1778.473669


In [ ]:
# ============================================================
# STAGE 5 — ENVIRONMENTAL STATE CONSTRUCTION
# Cell 1: Inspect actual ocean_df distributions
#
# IMPORTANT:
# These statistics come directly from the uploaded dataset.
# No physical thresholds are being invented here.
# ============================================================

environmental_columns = [
    "depth_m",
    "temperature_C",
    "salinity_PSU",
    "turbidity_NTU",
    "c_mackenzie_mps"
]

print("ENVIRONMENTAL DATA DISTRIBUTION")
print("=" * 75)

display(
    ocean_df[environmental_columns]
    .describe()
    .T
)

print("\nMissing values:")
display(
    ocean_df[environmental_columns]
    .isna()
    .sum()
)

print("\nSelected quantiles:")
display(
    ocean_df[environmental_columns]
    .quantile([0.10, 0.25, 0.50, 0.75, 0.90])
)

ENVIRONMENTAL DATA DISTRIBUTION


,count,mean,std,min,25%,50%,75%,max
depth_m,1839.0,244.186786,506.620113,0.500000,10.000000,50.000000,200.000000,5000.000000
temperature_C,1839.0,18.237474,7.214249,1.074628,12.992215,19.498948,23.780825,35.208053
salinity_PSU,1839.0,34.950096,3.053257,4.504050,34.935500,35.695660,36.267600,39.003450
turbidity_NTU,1839.0,1.003337,3.677794,0.095900,0.633350,0.727600,0.825700,99.891000
c_mackenzie_mps,1839.0,1518.340730,16.783648,1473.994739,1504.153614,1521.457548,1532.155003,1558.807995



Missing values:


,0
depth_m,0
temperature_C,0
salinity_PSU,0
turbidity_NTU,0
c_mackenzie_mps,0



Selected quantiles:


,depth_m,temperature_C,salinity_PSU,turbidity_NTU,c_mackenzie_mps
0.10,2.0,6.476330,33.82001,0.34246,1493.154582
0.25,10.0,12.992215,34.93550,0.63335,1504.153614
0.50,50.0,19.498948,35.69566,0.72760,1521.457548
0.75,200.0,23.780825,36.26760,0.82570,1532.155003
0.90,750.0,27.767300,36.43177,1.20192,1539.420043


In [ ]:
# ============================================================
# STAGE 5 — Cell 2
# Data-driven environmental state quantization
# ============================================================

# ------------------------------------------------------------
# Quantile state function
# ------------------------------------------------------------

def make_quantile_state(series, prefix):
    """
    Convert a continuous environmental variable into
    four data-driven states.

    Q1/Q2/Q3/Q4 are statistical bins only.
    They are NOT physical classifications.
    """

    # rank(method='first') avoids duplicate-edge problems
    # when many observations have identical values.
    ranked = series.rank(method="first")

    return pd.qcut(
        ranked,
        q=4,
        labels=[
            f"{prefix}_Q1",
            f"{prefix}_Q2",
            f"{prefix}_Q3",
            f"{prefix}_Q4"
        ]
    )


# ------------------------------------------------------------
# Create state labels directly inside ocean_df
# ------------------------------------------------------------

ocean_df["depth_state"] = make_quantile_state(
    ocean_df["depth_m"],
    "DEPTH"
)

ocean_df["temperature_state"] = make_quantile_state(
    ocean_df["temperature_C"],
    "TEMP"
)

ocean_df["salinity_state"] = make_quantile_state(
    ocean_df["salinity_PSU"],
    "SAL"
)

ocean_df["turbidity_state"] = make_quantile_state(
    ocean_df["turbidity_NTU"],
    "TURB"
)

# ------------------------------------------------------------
# Compact environmental state ID
# ------------------------------------------------------------

ocean_df["environment_state_id"] = (
    ocean_df["depth_state"].astype(str)
    + "_"
    + ocean_df["temperature_state"].astype(str)
    + "_"
    + ocean_df["salinity_state"].astype(str)
    + "_"
    + ocean_df["turbidity_state"].astype(str)
)

# Number of observations in every state
state_counts = (
    ocean_df["environment_state_id"]
    .value_counts()
    .rename_axis("environment_state_id")
    .reset_index(name="observations")
)

print("ENVIRONMENTAL STATE CONSTRUCTION")
print("=" * 75)

print(
    "Original ocean observations :",
    len(ocean_df)
)

print(
    "Unique environmental states :",
    ocean_df["environment_state_id"].nunique()
)

print("\nState-frequency statistics:")
display(
    state_counts["observations"].describe()
)

print("\nMost populated states:")
display(
    state_counts.head(10)
)

print("\nExample state assignments:")
display(
    ocean_df[
        [
            "depth_m",
            "temperature_C",
            "salinity_PSU",
            "turbidity_NTU",
            "environment_state_id"
        ]
    ].head(10)
)

ENVIRONMENTAL STATE CONSTRUCTION
Original ocean observations : 1839
Unique environmental states : 130

State-frequency statistics:


,observations
count,130.000000
mean,14.146154
std,17.759422
min,1.000000
25%,3.000000
50%,8.000000
75%,20.000000
max,112.000000



Most populated states:


,environment_state_id,observations
0,DEPTH_Q4_TEMP_Q1_SAL_Q1_TURB_Q2,112
1,DEPTH_Q1_TEMP_Q4_SAL_Q1_TURB_Q4,92
2,DEPTH_Q4_TEMP_Q1_SAL_Q2_TURB_Q2,86
3,DEPTH_Q4_TEMP_Q1_SAL_Q1_TURB_Q3,67
4,DEPTH_Q3_TEMP_Q3_SAL_Q4_TURB_Q1,45
5,DEPTH_Q4_TEMP_Q1_SAL_Q2_TURB_Q1,45
6,DEPTH_Q4_TEMP_Q1_SAL_Q2_TURB_Q3,43
7,DEPTH_Q2_TEMP_Q4_SAL_Q3_TURB_Q1,43
8,DEPTH_Q4_TEMP_Q1_SAL_Q2_TURB_Q4,40
9,DEPTH_Q3_TEMP_Q2_SAL_Q3_TURB_Q2,36



Example state assignments:


,depth_m,temperature_C,salinity_PSU,turbidity_NTU,environment_state_id
0,1.0,14.41920,24.17840,0.55410,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1
1,2.0,14.51465,34.82145,0.52380,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1
2,5.0,14.26420,35.93365,0.74440,DEPTH_Q1_TEMP_Q2_SAL_Q3_TURB_Q3
3,10.0,14.11650,35.96055,0.85240,DEPTH_Q1_TEMP_Q2_SAL_Q3_TURB_Q4
4,20.0,14.07870,35.98490,1.26050,DEPTH_Q2_TEMP_Q2_SAL_Q3_TURB_Q4
5,1.0,16.83100,25.05710,0.57535,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1
6,2.0,16.64970,34.94135,0.55470,DEPTH_Q1_TEMP_Q2_SAL_Q2_TURB_Q1
7,5.0,15.75525,36.10045,0.55770,DEPTH_Q1_TEMP_Q2_SAL_Q3_TURB_Q1
8,10.0,15.37495,36.06810,0.56575,DEPTH_Q1_TEMP_Q2_SAL_Q3_TURB_Q1
9,20.0,15.15875,36.05090,0.75840,DEPTH_Q2_TEMP_Q2_SAL_Q3_TURB_Q3


In [ ]:
# ============================================================
# STAGE 5 — Cell 3
# Build representative environmental state table
#
# Each populated state is represented by the median of the
# original observations belonging to that state.
#
# This is statistical compression, NOT a new environmental
# dataset or a replacement for ocean_df.
# ============================================================

state_columns = [
    "environment_state_id",
    "depth_m",
    "temperature_C",
    "salinity_PSU",
    "turbidity_NTU",
    "c_mackenzie_mps"
]

environment_states_df = (
    ocean_df[state_columns]
    .groupby("environment_state_id", observed=True)
    .median(numeric_only=True)
    .reset_index()
)

# Add observation count for traceability
state_observation_counts = (
    ocean_df["environment_state_id"]
    .value_counts()
    .rename_axis("environment_state_id")
    .reset_index(name="observations")
)

environment_states_df = environment_states_df.merge(
    state_observation_counts,
    on="environment_state_id",
    how="left"
)

print("REPRESENTATIVE ENVIRONMENTAL STATES")
print("=" * 75)

print(
    "Original observations :",
    len(ocean_df)
)

print(
    "Populated states      :",
    len(environment_states_df)
)

print("\nRepresentative-state table:")
display(
    environment_states_df.head(10)
)

print("\nObservation-count summary:")
display(
    environment_states_df["observations"].describe()
)

REPRESENTATIVE ENVIRONMENTAL STATES
Original observations : 1839
Populated states      : 130

Representative-state table:


,environment_state_id,depth_m,temperature_C,salinity_PSU,turbidity_NTU,c_mackenzie_mps,observations
0,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,3.0,11.676825,29.907525,0.761975,1489.552899,2
1,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,5.0,12.451406,35.029951,0.727600,1498.474219,2
2,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,3.5,11.847276,35.303154,0.745325,1496.719121,2
3,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1,1.0,16.602250,25.057100,0.404350,1500.745861,13
4,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q3,1.0,19.322400,12.199300,0.783700,1493.558159,1
5,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q4,1.0,18.904000,25.445900,1.810550,1501.859336,13
6,DEPTH_Q1_TEMP_Q2_SAL_Q2_TURB_Q1,2.0,16.649700,35.193100,0.554700,1511.755161,3
7,DEPTH_Q1_TEMP_Q2_SAL_Q2_TURB_Q2,5.0,16.534109,35.475625,0.727600,1512.079484,5
8,DEPTH_Q1_TEMP_Q2_SAL_Q2_TURB_Q3,5.0,16.777209,35.283996,0.727600,1512.626968,6
9,DEPTH_Q1_TEMP_Q2_SAL_Q2_TURB_Q4,5.0,17.555150,35.481400,0.846950,1515.345470,3



Observation-count summary:


,observations
count,130.000000
mean,14.146154
std,17.759422
min,1.000000
25%,3.000000
50%,8.000000
75%,20.000000
max,112.000000


In [ ]:
# ============================================================
# STAGE 5 — Cell 4
# Create environment × mission operating states
# ============================================================

# Cartesian product:
# 130 environmental states × 3 mission scenarios = 390 states

environment_states_df["_join_key"] = 1
scenario_df["_join_key"] = 1

operating_states_df = environment_states_df.merge(
    scenario_df,
    on="_join_key",
    how="inner"
).drop(columns="_join_key")

# Give every environment × mission combination a unique ID
operating_states_df.insert(
    0,
    "operating_state_id",
    [
        f"OP_{i:03d}"
        for i in range(1, len(operating_states_df) + 1)
    ]
)

print("ENVIRONMENT × MISSION OPERATING STATES")
print("=" * 75)

print(
    "Environmental states :",
    environment_states_df["environment_state_id"].nunique()
)

print(
    "Mission scenarios    :",
    scenario_df["scenario_id"].nunique()
)

print(
    "Operating states     :",
    len(operating_states_df)
)

print("\nExpected operating states:")
print(
    environment_states_df["environment_state_id"].nunique()
    * scenario_df["scenario_id"].nunique()
)

print("\nOperating-state distribution:")
display(
    operating_states_df[
        [
            "operating_state_id",
            "environment_state_id",
            "scenario_id",
            "depth_m",
            "temperature_C",
            "salinity_PSU",
            "turbidity_NTU",
            "c_mackenzie_mps",
            "range_m"
        ]
    ].head(12)
)

ENVIRONMENT × MISSION OPERATING STATES
Environmental states : 130
Mission scenarios    : 3
Operating states     : 390

Expected operating states:
390

Operating-state distribution:


,operating_state_id,environment_state_id,scenario_id,depth_m,temperature_C,salinity_PSU,turbidity_NTU,c_mackenzie_mps,range_m
0,OP_001,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S1_Near_LowNoise,3.0,11.676825,29.907525,0.761975,1489.552899,100
1,OP_002,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S2_Medium,3.0,11.676825,29.907525,0.761975,1489.552899,500
2,OP_003,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S3_Far_HighNoise,3.0,11.676825,29.907525,0.761975,1489.552899,1000
3,OP_004,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S1_Near_LowNoise,5.0,12.451406,35.029951,0.727600,1498.474219,100
4,OP_005,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S2_Medium,5.0,12.451406,35.029951,0.727600,1498.474219,500
5,OP_006,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S3_Far_HighNoise,5.0,12.451406,35.029951,0.727600,1498.474219,1000
6,OP_007,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S1_Near_LowNoise,3.5,11.847276,35.303154,0.745325,1496.719121,100
7,OP_008,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S2_Medium,3.5,11.847276,35.303154,0.745325,1496.719121,500
8,OP_009,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S3_Far_HighNoise,3.5,11.847276,35.303154,0.745325,1496.719121,1000
9,OP_010,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1,S1_Near_LowNoise,1.0,16.602250,25.057100,0.404350,1500.745861,100


In [ ]:
# ============================================================
# STAGE 6 — CANDIDATE × OPERATING-STATE EVALUATION
# Cell 1: Build the complete evaluation matrix
#
# 675 waveform candidates × 390 operating states
# = 263,250 evaluations
#
# This replaces the earlier generic candidate × scenario
# detection evaluation for final LUT optimization.
# ============================================================

# ------------------------------------------------------------
# Candidate columns required for evaluation
# ------------------------------------------------------------

candidate_eval = candidate_df[
    [
        "candidate_id",
        "waveform_type",
        "fc_Hz",
        "fc_kHz",
        "B_Hz",
        "B_kHz",
        "Tp_us",
        "Tp_s",
        "TBP",
        "f_low_Hz",
        "f_high_Hz",
        "resolution_valid",
        "frequency_valid_final",
        "sampling_valid_final"
    ]
].copy()

# ------------------------------------------------------------
# Operating-state columns required for evaluation
# ------------------------------------------------------------

state_eval = operating_states_df[
    [
        "operating_state_id",
        "environment_state_id",
        "scenario_id",
        "depth_m",
        "temperature_C",
        "salinity_PSU",
        "turbidity_NTU",
        "c_mackenzie_mps",
        "range_m",
        "ambient_noise_dB",
        "target_strength_dB",
        "transmit_level_dB",
        "detection_threshold_dB"
    ]
].copy()

# Cartesian product
candidate_eval["_join_key"] = 1
state_eval["_join_key"] = 1

evaluation_df = candidate_eval.merge(
    state_eval,
    on="_join_key",
    how="inner"
).drop(columns="_join_key")

print("CANDIDATE × OPERATING-STATE MATRIX")
print("=" * 75)

print("Waveform candidates :", len(candidate_df))
print("Operating states    :", len(operating_states_df))
print("Expected evaluations:",
      len(candidate_df) * len(operating_states_df))
print("Actual evaluations  :", len(evaluation_df))

display(
    evaluation_df.head(10)
)

CANDIDATE × OPERATING-STATE MATRIX
Waveform candidates : 675
Operating states    : 390
Expected evaluations: 263250
Actual evaluations  : 263250


,candidate_id,waveform_type,fc_Hz,fc_kHz,B_Hz,B_kHz,Tp_us,Tp_s,TBP,f_low_Hz,...,depth_m,temperature_C,salinity_PSU,turbidity_NTU,c_mackenzie_mps,range_m,ambient_noise_dB,target_strength_dB,transmit_level_dB,detection_threshold_dB
0,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,3.0,11.676825,29.907525,0.761975,1489.552899,100,50,10,180,10
1,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,3.0,11.676825,29.907525,0.761975,1489.552899,500,65,0,180,10
2,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,3.0,11.676825,29.907525,0.761975,1489.552899,1000,80,-10,180,10
3,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,5.0,12.451406,35.029951,0.727600,1498.474219,100,50,10,180,10
4,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,5.0,12.451406,35.029951,0.727600,1498.474219,500,65,0,180,10
5,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,5.0,12.451406,35.029951,0.727600,1498.474219,1000,80,-10,180,10
6,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,3.5,11.847276,35.303154,0.745325,1496.719121,100,50,10,180,10
7,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,3.5,11.847276,35.303154,0.745325,1496.719121,500,65,0,180,10
8,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,3.5,11.847276,35.303154,0.745325,1496.719121,1000,80,-10,180,10
9,1,LFM,50000,50,857.344397,0.857344,100,0.0001,0.085734,49571.327801,...,1.0,16.602250,25.057100,0.404350,1500.745861,100,50,10,180,10


In [ ]:
# ============================================================
# STAGE 6 — Cell 2
# State-specific resolution, propagation and detection
# ============================================================

# ------------------------------------------------------------
# 1. Range resolution
# ------------------------------------------------------------

evaluation_df["range_resolution_m"] = (
    evaluation_df["c_mackenzie_mps"]
    /
    (2 * evaluation_df["B_Hz"])
)

evaluation_df["resolution_valid"] = (
    evaluation_df["range_resolution_m"]
    <= REQUIRED_RANGE_RESOLUTION_M
)

# ------------------------------------------------------------
# 2. Propagation loss
# ------------------------------------------------------------

evaluation_df["TL_proxy_dB"] = [
    transmission_loss_proxy_db(fc, rng)
    for fc, rng in zip(
        evaluation_df["fc_kHz"],
        evaluation_df["range_m"]
    )
]

# ------------------------------------------------------------
# 3. Pulse-compression / processing gain
# ------------------------------------------------------------

evaluation_df["processing_gain_dB"] = (
    10
    *
    np.log10(
        np.maximum(
            evaluation_df["TBP"],
            1.0
        )
    )
)

# ------------------------------------------------------------
# 4. Detection-excess proxy
# ------------------------------------------------------------
#
# SE_proxy =
# SL + PG + TS - 2TL - NL - DT
#
# All source terms other than environmental state variables
# are currently unvalidated scenario assumptions.

evaluation_df["SE_proxy_dB"] = (
    evaluation_df["transmit_level_dB"]
    + evaluation_df["processing_gain_dB"]
    + evaluation_df["target_strength_dB"]
    - 2 * evaluation_df["TL_proxy_dB"]
    - evaluation_df["ambient_noise_dB"]
    - evaluation_df["detection_threshold_dB"]
)

# ------------------------------------------------------------
# 5. Detection feasibility
# ------------------------------------------------------------

evaluation_df["detection_valid"] = (
    evaluation_df["SE_proxy_dB"] >= SE_MIN_DB
)

# ------------------------------------------------------------
# 6. Hardware / sampling feasibility
# ------------------------------------------------------------

evaluation_df["hardware_valid"] = (
    evaluation_df["frequency_valid_final"]
    &
    evaluation_df["sampling_valid_final"]
)

# ------------------------------------------------------------
# 7. Overall hard feasibility
# ------------------------------------------------------------

evaluation_df["hard_feasible"] = (
    evaluation_df["resolution_valid"]
    &
    evaluation_df["hardware_valid"]
    &
    evaluation_df["detection_valid"]
)

print("STATE-SPECIFIC PHYSICS EVALUATION")
print("=" * 75)

print(
    "Resolution-valid:",
    evaluation_df["resolution_valid"].sum()
)

print(
    "Detection-valid:",
    evaluation_df["detection_valid"].sum()
)

print(
    "Hardware-valid:",
    evaluation_df["hardware_valid"].sum()
)

print(
    "Hard-feasible:",
    evaluation_df["hard_feasible"].sum(),
    "/",
    len(evaluation_df)
)

print("\nHard-feasible evaluations by mission:")
display(
    evaluation_df
    .groupby("scenario_id")["hard_feasible"]
    .agg(["sum", "count"])
    .reset_index()
)

print("\nDetection proxy by mission:")
display(
    evaluation_df
    .groupby("scenario_id")["SE_proxy_dB"]
    .agg(["min", "mean", "median", "max"])
    .reset_index()
)

STATE-SPECIFIC PHYSICS EVALUATION
Resolution-valid: 263250
Detection-valid: 91260
Hardware-valid: 204750
Hard-feasible: 70980 / 263250

Hard-feasible evaluations by mission:


,scenario_id,sum,count
0,S1_Near_LowNoise,68250,87750
1,S2_Medium,2730,87750
2,S3_Far_HighNoise,0,87750



Detection proxy by mission:


,scenario_id,min,mean,median,max
0,S1_Near_LowNoise,49.760849,53.611741,52.158968,64.373396
1,S2_Medium,-4.154555,0.311664,-1.531374,11.375968
2,S3_Far_HighNoise,-42.391510,-37.156132,-38.827247,-25.713516


In [ ]:
# ============================================================
# STAGE 7A — JOULE-AWARE WAVEFORM METRICS
# ============================================================
# Purpose:
# Convert detection margin into a relative amplitude requirement
# and pulse-energy proxy for every hard-feasible evaluation.
#
# IMPORTANT:
# Absolute Joule energy is NOT claimed here because the actual
# transducer efficiency, impedance, voltage/current limits and
# acoustic output calibration are not yet available.
#
# Therefore:
#   energy_proxy = relative amplitude^2 × pulse duration
#
# This is used only for comparative optimization.
# ============================================================

energy_df = evaluation_df.copy()

# Detection margin above the required threshold
energy_df["detection_margin_dB"] = (
    energy_df["SE_proxy_dB"] - SE_MIN_DB
)

# Normalized amplitude required to achieve SE_MIN_DB
#
# If candidate already has large detection margin:
#     required amplitude < 1
#
# If candidate is exactly at threshold:
#     required amplitude = 1
#
# Only hard-feasible candidates are retained later.
energy_df["amplitude_required_normalized"] = np.power(
    10.0,
    (SE_MIN_DB - energy_df["SE_proxy_dB"]) / 20.0
)

# Relative pulse-energy proxy
#
# Energy ∝ A² × Tp
#
# Tp is expressed in microseconds so the value remains easy
# to interpret. It is NOT an absolute Joule value.
energy_df["energy_proxy"] = (
    energy_df["amplitude_required_normalized"] ** 2
    * energy_df["Tp_us"]
)

# Keep only candidates that satisfy all hard constraints
hard_feasible_df = energy_df[
    energy_df["hard_feasible"]
].copy()

print("=" * 75)
print("STAGE 7A — JOULE-AWARE METRICS")
print("=" * 75)

print(f"Total evaluations      : {len(energy_df):,}")
print(f"Hard-feasible          : {len(hard_feasible_df):,}")

if len(hard_feasible_df) > 0:
    print("\nEnergy proxy statistics (hard-feasible only):")
    print(
        hard_feasible_df["energy_proxy"]
        .describe()
        .round(4)
    )

    print("\nAmplitude requirement statistics:")
    print(
        hard_feasible_df["amplitude_required_normalized"]
        .describe()
        .round(4)
    )

STAGE 7A — JOULE-AWARE METRICS
Total evaluations      : 263,250
Hard-feasible          : 70,980

Energy proxy statistics (hard-feasible only):
count    70980.0000
mean        60.5224
std        303.2211
min          0.0073
25%          0.0101
50%          0.0176
75%          0.0502
max       1778.4737
Name: energy_proxy, dtype: float64

Amplitude requirement statistics:
count    70980.0000
mean         0.0411
std          0.1692
min          0.0019
25%          0.0054
50%          0.0082
75%          0.0101
max          0.9430
Name: amplitude_required_normalized, dtype: float64


In [ ]:
# ============================================================
# STAGE 7A — WAVEFORM QUALITY / DESIGN METRICS
# ============================================================

# Processing gain
hard_feasible_df["processing_gain_dB"] = (
    10 * np.log10(
        np.maximum(hard_feasible_df["TBP"], 1.0)
    )
)

# Resolution margin:
# How much better than the required 1 m resolution?
hard_feasible_df["resolution_margin_m"] = (
    REQUIRED_RANGE_RESOLUTION_M
    - hard_feasible_df["range_resolution_m"]
)

# Resolution improvement factor
#
# 1.0  -> exactly 1 m
# >1   -> finer than required
hard_feasible_df["resolution_factor"] = (
    REQUIRED_RANGE_RESOLUTION_M
    / hard_feasible_df["range_resolution_m"]
)

# Bandwidth utilization relative to the minimum required
hard_feasible_df["bandwidth_factor"] = (
    hard_feasible_df["B_Hz"]
    / hard_feasible_df["c_mackenzie_mps"].div(
        2 * REQUIRED_RANGE_RESOLUTION_M
    )
)

print("=" * 75)
print("WAVEFORM QUALITY METRICS")
print("=" * 75)

print("\nResolution:")
print(
    hard_feasible_df["range_resolution_m"]
    .describe()
    .round(4)
)

print("\nProcessing gain (dB):")
print(
    hard_feasible_df["processing_gain_dB"]
    .describe()
    .round(4)
)

print("\nBandwidth factor:")
print(
    hard_feasible_df["bandwidth_factor"]
    .describe()
    .round(3)
)

WAVEFORM QUALITY METRICS

Resolution:
count    70980.0000
mean         0.3327
std          0.3017
min          0.0542
25%          0.1093
50%          0.2216
75%          0.4462
max          0.9013
Name: range_resolution_m, dtype: float64

Processing gain (dB):
count    70980.0000
mean         4.1081
std          4.6773
min          0.0000
25%          0.0000
50%          2.3419
75%          8.3625
max         14.3831
Name: processing_gain_dB, dtype: float64

Bandwidth factor:
count    70980.000
mean         7.415
std          6.395
min          1.109
25%          2.241
50%          4.512
75%          9.151
max         18.445
Name: bandwidth_factor, dtype: float64


In [ ]:
# ============================================================
# STAGE 7A — FEASIBLE CANDIDATE DISTRIBUTION
# ============================================================

family_summary = (
    hard_feasible_df
    .groupby("waveform_type")
    .agg(
        feasible_evaluations=("candidate_id", "size"),
        unique_candidates=("candidate_id", "nunique"),
        mean_energy_proxy=("energy_proxy", "mean"),
        median_energy_proxy=("energy_proxy", "median"),
        min_energy_proxy=("energy_proxy", "min"),
        mean_detection_margin_dB=("detection_margin_dB", "mean"),
        mean_resolution_m=("range_resolution_m", "mean")
    )
    .reset_index()
)

print("=" * 75)
print("FEASIBLE CANDIDATES BY WAVEFORM FAMILY")
print("=" * 75)

display(
    family_summary.round(4)
)

FEASIBLE CANDIDATES BY WAVEFORM FAMILY


,waveform_type,feasible_evaluations,unique_candidates,mean_energy_proxy,median_energy_proxy,min_energy_proxy,mean_detection_margin_dB,mean_resolution_m
0,Geometric_NFM,23660,175,60.5224,0.0176,0.0073,41.9856,0.3327
1,LFM,23660,175,60.5224,0.0176,0.0073,41.9856,0.3327
2,Phase_Coded,23660,175,60.5224,0.0176,0.0073,41.9856,0.3327


In [ ]:
# ============================================================
# STAGE 7B — PHYSICALLY CONSISTENT WAVEFORM GENERATION
# ============================================================
#
# We now evaluate the waveform in COMPLEX BASEBAND.
#
# Why?
# The waveform-quality metrics (ACF, PSL, ISL) depend on the
# modulation envelope, not on the 50-600 kHz carrier itself.
#
# This also makes the candidate bandwidth B explicit.
#
# Families:
#   LFM
#   Geometric_NFM
#   Phase_Coded
#
# fs = 2 MHz remains the actual DAC/transmit sampling rate.
# ============================================================

def generate_lfm_baseband(B_hz, Tp_s, fs):
    """
    Complex-baseband LFM.

    Instantaneous frequency:
        -B/2 -> +B/2
    """

    t = np.arange(
        0,
        Tp_s,
        1.0 / fs
    )

    k = B_hz / Tp_s

    phase = 2.0 * np.pi * (
        -0.5 * B_hz * t
        + 0.5 * k * t**2
    )

    return np.exp(1j * phase)


def generate_geometric_nfm_baseband(B_hz, Tp_s, fs):
    """
    Complex-baseband geometric/nonlinear FM.

    Frequency trajectory is exponential between
    approximately -B/2 and +B/2.

    A small offset is used because an exponential law
    cannot cross zero frequency directly.
    """

    t = np.arange(
        0,
        Tp_s,
        1.0 / fs
    )

    # Normalized time
    u = t / Tp_s

    # Smooth nonlinear frequency law.
    #
    # sinh() gives a symmetric nonlinear sweep while
    # remaining well-defined around zero.
    strength = 2.0

    inst_freq = (
        (B_hz / 2.0)
        * np.sinh(
            strength * (2.0 * u - 1.0)
        )
        / np.sinh(strength)
    )

    # Numerical integration of instantaneous frequency.
    phase = 2.0 * np.pi * np.cumsum(
        inst_freq
    ) / fs

    return np.exp(1j * phase)


def generate_phase_coded_baseband(
    B_hz,
    Tp_s,
    fs
):
    """
    Bandwidth-consistent phase-coded waveform.

    Chip duration is approximately 1/B.

    Therefore:
        N_chips ≈ B * Tp = TBP

    If TBP < 2, a meaningful multi-chip phase-coded
    waveform cannot be constructed, so this candidate
    is marked invalid for phase-coded waveform-quality
    analysis.
    """

    TBP = B_hz * Tp_s

    N_chips = int(
        np.floor(TBP)
    )

    if N_chips < 2:
        return None

    # Barker-13 when enough chips exist.
    barker13 = np.array([
        1, 1, 1, 1, 1, -1, -1,
        1, 1, -1, 1, -1, 1
    ])

    if N_chips <= 13:

        # Use the first N chips of Barker-13.
        code = barker13[:N_chips]

    else:

        # For longer codes, repeat the Barker sequence.
        repetitions = int(
            np.ceil(
                N_chips / len(barker13)
            )
        )

        code = np.tile(
            barker13,
            repetitions
        )[:N_chips]

    # --------------------------------------------------------
    # Exactly match the requested pulse duration.
    # --------------------------------------------------------

    t = np.arange(
        0,
        Tp_s,
        1.0 / fs
    )

    samples_per_chip = len(t) / N_chips

    chip_index = np.floor(
        np.arange(len(t))
        / samples_per_chip
    ).astype(int)

    chip_index = np.minimum(
        chip_index,
        N_chips - 1
    )

    phase_code = code[chip_index]

    return phase_code.astype(complex)


def generate_baseband_waveform(row):
    """
    Generate the correct waveform family for one candidate.
    """

    B = row["B_Hz"]
    Tp = row["Tp_s"]

    if row["waveform_type"] == "LFM":

        waveform = generate_lfm_baseband(
            B,
            Tp,
            FS
        )

    elif row["waveform_type"] == "Geometric_NFM":

        waveform = generate_geometric_nfm_baseband(
            B,
            Tp,
            FS
        )

    elif row["waveform_type"] == "Phase_Coded":

        waveform = generate_phase_coded_baseband(
            B,
            Tp,
            FS
        )

    else:

        raise ValueError(
            f"Unknown waveform family: "
            f"{row['waveform_type']}"
        )

    # Invalid phase-coded candidate
    if waveform is None:
        return None

    # Unit peak normalization
    peak = np.max(
        np.abs(waveform)
    )

    if peak > 0:
        waveform = waveform / peak

    return waveform

In [ ]:
# ============================================================
# STAGE 7B — CORRECTED ACF / PSL / ISL
# ============================================================

def calculate_acf_metrics(
    waveform,
    B_hz,
    Tp_s,
    fs
):
    """
    Calculate normalized autocorrelation metrics.

    PSL:
        Peak sidelobe level relative to main peak.

    ISL:
        Integrated sidelobe level relative to mainlobe energy.

    For TBP < 2:
        sidelobe metrics are marked invalid because there is
        insufficient time-bandwidth product for a meaningful
        pulse-compression sidelobe measurement.
    """

    TBP = B_hz * Tp_s

    # --------------------------------------------------------
    # Low-TBP candidates
    # --------------------------------------------------------

    if TBP < 2.0:

        return {
            "TBP_waveform": TBP,
            "waveform_quality_valid": False,
            "psl_dB": np.nan,
            "isl_dB": np.nan,
            "mainlobe_width_us": np.nan,
            "mainlobe_width_samples": np.nan
        }

    x = np.asarray(waveform)

    N = len(x)

    # --------------------------------------------------------
    # FFT autocorrelation
    # --------------------------------------------------------

    nfft = 2 ** int(
        np.ceil(
            np.log2(
                2 * N - 1
            )
        )
    )

    X = np.fft.fft(
        x,
        n=nfft
    )

    acf = np.fft.ifft(
        X * np.conj(X)
    )

    acf = np.concatenate([
        acf[-(N - 1):],
        acf[:N]
    ])

    acf_mag = np.abs(acf)

    zero_index = N - 1

    peak = acf_mag[zero_index]

    if peak <= 0:

        return {
            "TBP_waveform": TBP,
            "waveform_quality_valid": False,
            "psl_dB": np.nan,
            "isl_dB": np.nan,
            "mainlobe_width_us": np.nan,
            "mainlobe_width_samples": np.nan
        }

    acf_norm = acf_mag / peak

    # --------------------------------------------------------
    # Mainlobe exclusion.
    #
    # The first-null scale is approximately 1/B.
    #
    # We use +/- 0.5/B as the nominal compressed mainlobe
    # half-width, giving approximately 1/B total width.
    # --------------------------------------------------------

    half_width = max(
        1,
        int(
            np.ceil(
                fs / (2.0 * B_hz)
            )
        )
    )

    left = zero_index - half_width
    right = zero_index + half_width

    left = max(0, left)
    right = min(
        len(acf_norm) - 1,
        right
    )

    mainlobe = acf_norm[
        left:right + 1
    ]

    sidelobes = np.concatenate([
        acf_norm[:left],
        acf_norm[right + 1:]
    ])

    # --------------------------------------------------------
    # PSL
    # --------------------------------------------------------

    if len(sidelobes) == 0:

        psl_dB = np.nan

    else:

        peak_sidelobe = np.max(
            sidelobes
        )

        if peak_sidelobe <= 0:
            psl_dB = -np.inf
        else:
            psl_dB = (
                20.0
                * np.log10(
                    peak_sidelobe
                )
            )

    # --------------------------------------------------------
    # ISL
    #
    # Ratio:
    #       sidelobe energy / mainlobe energy
    # --------------------------------------------------------

    main_energy = np.sum(
        mainlobe ** 2
    )

    sidelobe_energy = np.sum(
        sidelobes ** 2
    )

    if (
        main_energy > 0
        and sidelobe_energy > 0
    ):

        isl_dB = (
            10.0
            * np.log10(
                sidelobe_energy
                / main_energy
            )
        )

    else:

        isl_dB = np.nan

    # --------------------------------------------------------
    # Nominal compressed-pulse width
    # --------------------------------------------------------

    mainlobe_width_samples = (
        right - left + 1
    )

    mainlobe_width_us = (
        mainlobe_width_samples
        / fs
        * 1e6
    )

    return {
        "TBP_waveform": TBP,
        "waveform_quality_valid": True,
        "psl_dB": psl_dB,
        "isl_dB": isl_dB,
        "mainlobe_width_us": mainlobe_width_us,
        "mainlobe_width_samples":
            mainlobe_width_samples
    }


# ============================================================
# Evaluate all 675 unique candidates.
# ============================================================

waveform_metric_rows = []

for _, row in unique_candidates.iterrows():

    waveform = generate_baseband_waveform(
        row
    )

    if waveform is None:

        metrics = {
            "TBP_waveform":
                row["B_Hz"] * row["Tp_s"],
            "waveform_quality_valid":
                False,
            "psl_dB":
                np.nan,
            "isl_dB":
                np.nan,
            "mainlobe_width_us":
                np.nan,
            "mainlobe_width_samples":
                np.nan
        }

    else:

        metrics = calculate_acf_metrics(
            waveform=waveform,
            B_hz=row["B_Hz"],
            Tp_s=row["Tp_s"],
            fs=FS
        )

    waveform_metric_rows.append({

        "candidate_id":
            row["candidate_id"],

        "waveform_type":
            row["waveform_type"],

        "fc_Hz":
            row["fc_Hz"],

        "B_Hz":
            row["B_Hz"],

        "Tp_us":
            row["Tp_us"],

        **metrics
    })


waveform_metrics_df = pd.DataFrame(
    waveform_metric_rows
)


print("=" * 75)
print("STAGE 7B — WAVEFORM QUALITY METRICS")
print("=" * 75)

print(
    f"Unique candidates evaluated: "
    f"{len(waveform_metrics_df)}"
)

print(
    f"Quality-valid candidates: "
    f"{waveform_metrics_df['waveform_quality_valid'].sum()}"
)

print(
    f"Quality-invalid candidates: "
    f"{(~waveform_metrics_df['waveform_quality_valid']).sum()}"
)

display(
    waveform_metrics_df.head(10).round(4)
)

STAGE 7B — WAVEFORM QUALITY METRICS
Unique candidates evaluated: 675
Quality-valid candidates: 270
Quality-invalid candidates: 405


,candidate_id,waveform_type,fc_Hz,B_Hz,Tp_us,TBP_waveform,waveform_quality_valid,psl_dB,isl_dB,mainlobe_width_us,mainlobe_width_samples
0,1,LFM,50000,857.3444,100,0.0857,False,NaN,NaN,NaN,NaN
1,2,LFM,50000,857.3444,200,0.1715,False,NaN,NaN,NaN,NaN
2,3,LFM,50000,857.3444,500,0.4287,False,NaN,NaN,NaN,NaN
3,4,LFM,50000,857.3444,1000,0.8573,False,NaN,NaN,NaN,NaN
4,5,LFM,50000,857.3444,2000,1.7147,False,NaN,NaN,NaN,NaN
5,6,LFM,50000,1714.6888,100,0.1715,False,NaN,NaN,NaN,NaN
6,7,LFM,50000,1714.6888,200,0.3429,False,NaN,NaN,NaN,NaN
7,8,LFM,50000,1714.6888,500,0.8573,False,NaN,NaN,NaN,NaN
8,9,LFM,50000,1714.6888,1000,1.7147,False,NaN,NaN,NaN,NaN
9,10,LFM,50000,1714.6888,2000,3.4294,True,-4.1735,-4.6423,584.5,1169.0


In [ ]:
# ============================================================
# STAGE 7B — VALIDATION
# ============================================================

print("=" * 75)
print("WAVEFORM FAMILY COMPARISON")
print("=" * 75)

family_waveform_summary = (
    waveform_metrics_df
    .groupby("waveform_type")
    .agg(

        candidates=(
            "candidate_id",
            "count"
        ),

        quality_valid=(
            "waveform_quality_valid",
            "sum"
        ),

        mean_psl_dB=(
            "psl_dB",
            "mean"
        ),

        median_psl_dB=(
            "psl_dB",
            "median"
        ),

        best_psl_dB=(
            "psl_dB",
            "min"
        ),

        mean_isl_dB=(
            "isl_dB",
            "mean"
        ),

        median_isl_dB=(
            "isl_dB",
            "median"
        ),

        mean_mainlobe_width_us=(
            "mainlobe_width_us",
            "mean"
        )
    )
    .reset_index()
)

display(
    family_waveform_summary.round(4)
)


# ------------------------------------------------------------
# TBP distribution by family
# ------------------------------------------------------------

tbp_summary = (
    waveform_metrics_df
    .groupby("waveform_type")
    .agg(

        min_TBP=(
            "TBP_waveform",
            "min"
        ),

        median_TBP=(
            "TBP_waveform",
            "median"
        ),

        max_TBP=(
            "TBP_waveform",
            "max"
        ),

        quality_valid=(
            "waveform_quality_valid",
            "sum"
        )
    )
    .reset_index()
)

print("\n" + "=" * 75)
print("TIME-BANDWIDTH PRODUCT CHECK")
print("=" * 75)

display(
    tbp_summary.round(4)
)


# ------------------------------------------------------------
# Bandwidth dependence
# ------------------------------------------------------------

bandwidth_check = (
    waveform_metrics_df
    .groupby(
        [
            "waveform_type",
            "B_Hz"
        ]
    )
    .agg(

        valid_candidates=(
            "waveform_quality_valid",
            "sum"
        ),

        mean_psl_dB=(
            "psl_dB",
            "mean"
        ),

        mean_isl_dB=(
            "isl_dB",
            "mean"
        ),

        mainlobe_width_us=(
            "mainlobe_width_us",
            "median"
        )
    )
    .reset_index()
)

print("\n" + "=" * 75)
print("BANDWIDTH DEPENDENCE")
print("=" * 75)

display(
    bandwidth_check.round(4)
)

WAVEFORM FAMILY COMPARISON


,waveform_type,candidates,quality_valid,mean_psl_dB,median_psl_dB,best_psl_dB,mean_isl_dB,median_isl_dB,mean_mainlobe_width_us
0,Geometric_NFM,225,90,-2.7076,-2.6021,-3.1178,-2.2918,-2.5855,190.3
1,LFM,225,90,-4.1227,-4.0803,-4.3761,-4.8590,-4.9219,190.3
2,Phase_Coded,225,90,-2.9123,-2.1586,-5.6926,0.0470,1.5723,190.3



TIME-BANDWIDTH PRODUCT CHECK


,waveform_type,min_TBP,median_TBP,max_TBP,quality_valid
0,Geometric_NFM,0.0857,1.7147,27.435,90
1,LFM,0.0857,1.7147,27.435,90
2,Phase_Coded,0.0857,1.7147,27.435,90



BANDWIDTH DEPENDENCE


,waveform_type,B_Hz,valid_candidates,mean_psl_dB,mean_isl_dB,mainlobe_width_us
0,Geometric_NFM,857.3444,0,NaN,NaN,NaN
1,Geometric_NFM,1714.6888,9,-2.8613,-1.1768,584.5
2,Geometric_NFM,3429.3776,18,-2.7197,-1.8809,292.5
3,Geometric_NFM,6858.7552,27,-2.6592,-2.3321,146.5
4,Geometric_NFM,13717.5104,36,-2.6994,-2.7457,73.5
5,LFM,857.3444,0,NaN,NaN,NaN
6,LFM,1714.6888,9,-4.1735,-4.6423,584.5
7,LFM,3429.3776,18,-4.1013,-4.7835,292.5
8,LFM,6858.7552,27,-4.0865,-4.9603,146.5
9,LFM,13717.5104,36,-4.1479,-4.8749,73.5


In [ ]:
# ============================================================
# OJAS FINAL WAVEFORM LIBRARY
# Only ~20-30 waveforms are needed in the deployable LUT.
# ============================================================

final_waveforms = []

waveform_id = 1

# ------------------------------------------------------------
# 1. LFM: 8 deliberately selected operating points
# ------------------------------------------------------------
lfm_configs = [
    (75,  2, 1000),
    (100, 2, 1000),
    (150, 2, 1000),
    (200, 2, 1000),
    (300, 2, 1000),
    (400, 2, 1000),
    (500, 2, 1000),
    (600, 2, 1000),
]

for fc_khz, b_mult, tp_us in lfm_configs:
    final_waveforms.append({
        "waveform_id": f"W{waveform_id:02d}",
        "waveform_type": "LFM",
        "fc_kHz": fc_khz,
        "B_multiplier": b_mult,
        "Tp_us": tp_us
    })
    waveform_id += 1


# ------------------------------------------------------------
# 2. Geometric/NFM: 8 alternatives
# ------------------------------------------------------------
nfm_configs = [
    (75,  2, 1000),
    (100, 2, 1000),
    (150, 2, 1000),
    (200, 2, 1000),
    (300, 2, 1000),
    (400, 2, 1000),
    (500, 2, 1000),
    (600, 2, 1000),
]

for fc_khz, b_mult, tp_us in nfm_configs:
    final_waveforms.append({
        "waveform_id": f"W{waveform_id:02d}",
        "waveform_type": "Geometric_NFM",
        "fc_kHz": fc_khz,
        "B_multiplier": b_mult,
        "Tp_us": tp_us
    })
    waveform_id += 1


# ------------------------------------------------------------
# 3. Phase coded: 6 alternatives
# ------------------------------------------------------------
phase_configs = [
    (75,  4, 1000),
    (100, 4, 1000),
    (150, 4, 1000),
    (200, 4, 1000),
    (300, 4, 1000),
    (400, 4, 1000),
]

for fc_khz, b_mult, tp_us in phase_configs:
    final_waveforms.append({
        "waveform_id": f"W{waveform_id:02d}",
        "waveform_type": "Phase_Coded",
        "fc_kHz": fc_khz,
        "B_multiplier": b_mult,
        "Tp_us": tp_us
    })
    waveform_id += 1


# ------------------------------------------------------------
# 4. Robust fallback waveforms
# ------------------------------------------------------------
final_waveforms.extend([
    {
        "waveform_id": "W23",
        "waveform_type": "LFM",
        "fc_kHz": 200,
        "B_multiplier": 8,
        "Tp_us": 2000
    },
    {
        "waveform_id": "W24",
        "waveform_type": "Geometric_NFM",
        "fc_kHz": 300,
        "B_multiplier": 8,
        "Tp_us": 2000
    }
])

final_waveform_df = pd.DataFrame(final_waveforms)

print("FINAL WAVEFORM COUNT:", len(final_waveform_df))
display(final_waveform_df)

FINAL WAVEFORM COUNT: 24


,waveform_id,waveform_type,fc_kHz,B_multiplier,Tp_us
0,W01,LFM,75,2,1000
1,W02,LFM,100,2,1000
2,W03,LFM,150,2,1000
3,W04,LFM,200,2,1000
4,W05,LFM,300,2,1000
5,W06,LFM,400,2,1000
6,W07,LFM,500,2,1000
7,W08,LFM,600,2,1000
8,W09,Geometric_NFM,75,2,1000
9,W10,Geometric_NFM,100,2,1000


In [ ]:
# ============================================================
# Convert compact waveform definitions into physical values
# ============================================================

# Use representative sound speed for waveform-library design.
# State-specific resolution will be recalculated later.
C_REF = ocean_df["c_mackenzie_mps"].median()

final_waveform_df["B_min_Hz"] = C_REF / (
    2 * REQUIRED_RANGE_RESOLUTION_M
)

final_waveform_df["B_Hz"] = (
    final_waveform_df["B_min_Hz"]
    * 1.10
    * final_waveform_df["B_multiplier"]
)

final_waveform_df["B_kHz"] = (
    final_waveform_df["B_Hz"] / 1e3
)

final_waveform_df["Tp_s"] = (
    final_waveform_df["Tp_us"] * 1e-6
)

final_waveform_df["TBP"] = (
    final_waveform_df["B_Hz"]
    * final_waveform_df["Tp_s"]
)

final_waveform_df["f_low_Hz"] = (
    final_waveform_df["fc_kHz"] * 1e3
    - final_waveform_df["B_Hz"] / 2
)

final_waveform_df["f_high_Hz"] = (
    final_waveform_df["fc_kHz"] * 1e3
    + final_waveform_df["B_Hz"] / 2
)

final_waveform_df["frequency_valid"] = (
    (final_waveform_df["f_low_Hz"] >= FC_MIN_HZ) &
    (final_waveform_df["f_high_Hz"] <= FC_MAX_HZ)
)

final_waveform_df["sampling_valid"] = (
    FS_HZ >= 2 * final_waveform_df["f_high_Hz"]
)

final_waveform_df["hardware_valid"] = (
    final_waveform_df["frequency_valid"] &
    final_waveform_df["sampling_valid"]
)

display(final_waveform_df)

,waveform_id,waveform_type,fc_kHz,B_multiplier,Tp_us,B_min_Hz,B_Hz,B_kHz,Tp_s,TBP,f_low_Hz,f_high_Hz,frequency_valid,sampling_valid,hardware_valid
0,W01,LFM,75,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,74163.198349,75836.801651,True,True,True
1,W02,LFM,100,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,99163.198349,100836.801651,True,True,True
2,W03,LFM,150,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,149163.198349,150836.801651,True,True,True
3,W04,LFM,200,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,199163.198349,200836.801651,True,True,True
4,W05,LFM,300,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,299163.198349,300836.801651,True,True,True
5,W06,LFM,400,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,399163.198349,400836.801651,True,True,True
6,W07,LFM,500,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,499163.198349,500836.801651,True,True,True
7,W08,LFM,600,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,599163.198349,600836.801651,False,True,False
8,W09,Geometric_NFM,75,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,74163.198349,75836.801651,True,True,True
9,W10,Geometric_NFM,100,2,1000,760.728774,1673.603303,1.673603,0.001,1.673603,99163.198349,100836.801651,True,True,True


In [ ]:
# Show only columns that could be related to detection
[c for c in operating_states_df.columns
 if any(x in c.lower() for x in
        ["sl", "ts", "noise", "nl", "threshold", "dt"])]

['ambient_noise_dB',
 'detection_threshold_dB',
 'noise_status',
 'detection_threshold_status']

In [ ]:
# Show all operating-state columns
print(operating_states_df.columns.tolist())

['operating_state_id', 'environment_state_id', 'depth_m', 'temperature_C', 'salinity_PSU', 'turbidity_NTU', 'c_mackenzie_mps', 'observations', 'scenario_id', 'range_m', 'ambient_noise_dB', 'target_strength_dB', 'transmit_level_dB', 'detection_threshold_dB', 'noise_status', 'target_strength_status', 'transmit_level_status', 'detection_threshold_status']


In [ ]:
# ============================================================
# OJAS FINAL EVALUATION
# 24 waveforms × 390 operating states
# ============================================================

wf = final_waveform_df.copy()

# Cartesian product: every waveform against every operating state
evaluation_24_df = (
    operating_states_df.assign(_key=1)
    .merge(wf.assign(_key=1), on="_key")
    .drop(columns="_key")
)

# ------------------------------------------------------------
# 1. State-specific range resolution
# ------------------------------------------------------------
evaluation_24_df["range_resolution_m"] = (
    evaluation_24_df["c_mackenzie_mps"]
    / (2 * evaluation_24_df["B_Hz"])
)

evaluation_24_df["resolution_valid"] = (
    evaluation_24_df["range_resolution_m"]
    <= REQUIRED_RANGE_RESOLUTION_M
)

# ------------------------------------------------------------
# 2. Propagation-loss proxy
# ------------------------------------------------------------
fc_khz = evaluation_24_df["fc_kHz"]
range_m = evaluation_24_df["range_m"]

alpha_db_per_km = (
    0.02 + 0.00008 * (fc_khz ** 1.5)
)

evaluation_24_df["TL_proxy_dB"] = (
    20 * np.log10(range_m)
    + alpha_db_per_km * (range_m / 1000)
)

# ------------------------------------------------------------
# 3. Processing gain
# ------------------------------------------------------------
evaluation_24_df["processing_gain_dB"] = (
    10 * np.log10(
        np.maximum(evaluation_24_df["TBP"], 1)
    )
)

# ------------------------------------------------------------
# 4. Detection equation
#
# SE = SL + PG + TS - 2TL - NL - DT
# ------------------------------------------------------------
evaluation_24_df["SE_proxy_dB"] = (
    evaluation_24_df["transmit_level_dB"]
    + evaluation_24_df["processing_gain_dB"]
    + evaluation_24_df["target_strength_dB"]
    - 2 * evaluation_24_df["TL_proxy_dB"]
    - evaluation_24_df["ambient_noise_dB"]
    - evaluation_24_df["detection_threshold_dB"]
)

evaluation_24_df["detection_margin_dB"] = (
    evaluation_24_df["SE_proxy_dB"]
    - SE_MIN_DB
)

evaluation_24_df["detection_valid"] = (
    evaluation_24_df["SE_proxy_dB"] >= SE_MIN_DB
)

# ------------------------------------------------------------
# 5. Hardware validity
# ------------------------------------------------------------
evaluation_24_df["hardware_valid"] = (
    evaluation_24_df["frequency_valid"]
    & evaluation_24_df["sampling_valid"]
)

# ------------------------------------------------------------
# 6. Final hard feasibility
# ------------------------------------------------------------
evaluation_24_df["hard_feasible"] = (
    evaluation_24_df["resolution_valid"]
    & evaluation_24_df["hardware_valid"]
    & evaluation_24_df["detection_valid"]
)

print("Total evaluations:", len(evaluation_24_df))
print(
    "Expected:",
    len(operating_states_df) * len(final_waveform_df)
)

print(
    "Hard-feasible evaluations:",
    evaluation_24_df["hard_feasible"].sum()
)

print("\nFeasible evaluations by scenario:")

display(
    evaluation_24_df
    .groupby("scenario_id")["hard_feasible"]
    .agg(
        total="count",
        feasible="sum",
        feasible_fraction="mean"
    )
)

Total evaluations: 9360
Expected: 9360
Hard-feasible evaluations: 3380

Feasible evaluations by scenario:


,total,feasible,feasible_fraction
scenario_id,,,
S1_Near_LowNoise,3120,3120,1.000000
S2_Medium,3120,260,0.083333
S3_Far_HighNoise,3120,0,0.000000


In [ ]:
# ============================================================
# ENERGY PROXY
# ============================================================

evaluation_24_df["amplitude_required_normalized"] = (
    10 ** (
        (SE_MIN_DB - evaluation_24_df["SE_proxy_dB"]) / 20
    )
)

evaluation_24_df["energy_proxy"] = (
    evaluation_24_df["amplitude_required_normalized"] ** 2
    * evaluation_24_df["Tp_us"]
)

print("Energy proxy calculated.")

display(
    evaluation_24_df[
        [
            "operating_state_id",
            "scenario_id",
            "waveform_id",
            "waveform_type",
            "fc_kHz",
            "B_kHz",
            "Tp_us",
            "TBP",
            "range_resolution_m",
            "SE_proxy_dB",
            "detection_margin_dB",
            "energy_proxy",
            "hard_feasible"
        ]
    ].head(20)
)

Energy proxy calculated.


,operating_state_id,scenario_id,waveform_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,range_resolution_m,SE_proxy_dB,detection_margin_dB,energy_proxy,hard_feasible
0,OP_001,S1_Near_LowNoise,W01,LFM,75,1.673603,1000,1.673603,0.445014,52.222133,42.222133,0.059950,True
1,OP_001,S1_Near_LowNoise,W02,LFM,100,1.673603,1000,1.673603,0.445014,52.216525,42.216525,0.060027,True
2,OP_001,S1_Near_LowNoise,W03,LFM,150,1.673603,1000,1.673603,0.445014,52.203131,42.203131,0.060213,True
3,OP_001,S1_Near_LowNoise,W04,LFM,200,1.673603,1000,1.673603,0.445014,52.187270,42.187270,0.060433,True
4,OP_001,S1_Near_LowNoise,W05,LFM,300,1.673603,1000,1.673603,0.445014,52.149387,42.149387,0.060962,True
5,OP_001,S1_Near_LowNoise,W06,LFM,400,1.673603,1000,1.673603,0.445014,52.104525,42.104525,0.061595,True
6,OP_001,S1_Near_LowNoise,W07,LFM,500,1.673603,1000,1.673603,0.445014,52.053640,42.053640,0.062321,True
7,OP_001,S1_Near_LowNoise,W08,LFM,75,13.388826,2000,26.777653,0.055627,64.263333,54.263333,0.007494,True
8,OP_001,S1_Near_LowNoise,W09,Geometric_NFM,75,1.673603,1000,1.673603,0.445014,52.222133,42.222133,0.059950,True
9,OP_001,S1_Near_LowNoise,W10,Geometric_NFM,100,1.673603,1000,1.673603,0.445014,52.216525,42.216525,0.060027,True


In [ ]:
# ============================================================
# REVISE ONLY THE 4 WAVEFORM SLOTS NEEDED FOR S2 COVERAGE
# Keep the 24-waveform library compact.
# ============================================================

# W08: replace invalid 600-kHz LFM with low-frequency,
# high-TBP LFM for medium-range operation
final_waveform_df.loc[
    final_waveform_df["waveform_id"] == "W08",
    ["waveform_type", "fc_kHz", "B_multiplier", "Tp_us"]
] = ["LFM", 50, 16, 2000]

# W16: replace invalid 600-kHz NFM with low-frequency,
# high-TBP NFM for medium-range operation
final_waveform_df.loc[
    final_waveform_df["waveform_id"] == "W16",
    ["waveform_type", "fc_kHz", "B_multiplier", "Tp_us"]
] = ["Geometric_NFM", 50, 16, 2000]

# Recalculate physical parameters for W08/W16
mask = final_waveform_df["waveform_id"].isin(["W08", "W16"])

final_waveform_df.loc[mask, "B_min_Hz"] = (
    C_REF / (2 * REQUIRED_RANGE_RESOLUTION_M)
)

final_waveform_df.loc[mask, "B_Hz"] = (
    final_waveform_df.loc[mask, "B_min_Hz"]
    * 1.10
    * final_waveform_df.loc[mask, "B_multiplier"]
)

final_waveform_df.loc[mask, "B_kHz"] = (
    final_waveform_df.loc[mask, "B_Hz"] / 1e3
)

final_waveform_df.loc[mask, "Tp_s"] = (
    final_waveform_df.loc[mask, "Tp_us"] * 1e-6
)

final_waveform_df.loc[mask, "TBP"] = (
    final_waveform_df.loc[mask, "B_Hz"]
    * final_waveform_df.loc[mask, "Tp_s"]
)

final_waveform_df.loc[mask, "f_low_Hz"] = (
    final_waveform_df.loc[mask, "fc_kHz"] * 1e3
    - final_waveform_df.loc[mask, "B_Hz"] / 2
)

final_waveform_df.loc[mask, "f_high_Hz"] = (
    final_waveform_df.loc[mask, "fc_kHz"] * 1e3
    + final_waveform_df.loc[mask, "B_Hz"] / 2
)

final_waveform_df.loc[mask, "frequency_valid"] = (
    (final_waveform_df.loc[mask, "f_low_Hz"] >= FC_MIN_HZ) &
    (final_waveform_df.loc[mask, "f_high_Hz"] <= FC_MAX_HZ)
)

final_waveform_df.loc[mask, "sampling_valid"] = (
    FS_HZ >= 2 * final_waveform_df.loc[mask, "f_high_Hz"]
)

final_waveform_df.loc[mask, "hardware_valid"] = (
    final_waveform_df.loc[mask, "frequency_valid"] &
    final_waveform_df.loc[mask, "sampling_valid"]
)

display(
    final_waveform_df[
        ["waveform_id", "waveform_type", "fc_kHz",
         "B_kHz", "Tp_us", "TBP",
         "f_low_Hz", "f_high_Hz",
         "hardware_valid"]
    ]
)

,waveform_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,f_low_Hz,f_high_Hz,hardware_valid
0,W01,LFM,75,1.673603,1000,1.673603,74163.198349,75836.801651,True
1,W02,LFM,100,1.673603,1000,1.673603,99163.198349,100836.801651,True
2,W03,LFM,150,1.673603,1000,1.673603,149163.198349,150836.801651,True
3,W04,LFM,200,1.673603,1000,1.673603,199163.198349,200836.801651,True
4,W05,LFM,300,1.673603,1000,1.673603,299163.198349,300836.801651,True
5,W06,LFM,400,1.673603,1000,1.673603,399163.198349,400836.801651,True
6,W07,LFM,500,1.673603,1000,1.673603,499163.198349,500836.801651,True
7,W08,LFM,50,13.388826,2000,26.777653,43305.586789,56694.413211,False
8,W09,Geometric_NFM,75,1.673603,1000,1.673603,74163.198349,75836.801651,True
9,W10,Geometric_NFM,100,1.673603,1000,1.673603,99163.198349,100836.801651,True


In [ ]:
# ============================================================
# FIX W08 AND W16
# Keep high-TBP S2 waveforms, but make occupied band valid.
# ============================================================

for wid in ["W08", "W16"]:
    final_waveform_df.loc[
        final_waveform_df["waveform_id"] == wid,
        "fc_kHz"
    ] = 75

# Recalculate occupied-band limits
mask = final_waveform_df["waveform_id"].isin(["W08", "W16"])

final_waveform_df.loc[mask, "f_low_Hz"] = (
    final_waveform_df.loc[mask, "fc_kHz"] * 1e3
    - final_waveform_df.loc[mask, "B_Hz"] / 2
)

final_waveform_df.loc[mask, "f_high_Hz"] = (
    final_waveform_df.loc[mask, "fc_kHz"] * 1e3
    + final_waveform_df.loc[mask, "B_Hz"] / 2
)

final_waveform_df.loc[mask, "frequency_valid"] = (
    (final_waveform_df.loc[mask, "f_low_Hz"] >= FC_MIN_HZ) &
    (final_waveform_df.loc[mask, "f_high_Hz"] <= FC_MAX_HZ)
)

final_waveform_df.loc[mask, "sampling_valid"] = (
    FS_HZ >= 2 * final_waveform_df.loc[mask, "f_high_Hz"]
)

final_waveform_df.loc[mask, "hardware_valid"] = (
    final_waveform_df.loc[mask, "frequency_valid"] &
    final_waveform_df.loc[mask, "sampling_valid"]
)

display(
    final_waveform_df[
        ["waveform_id", "waveform_type", "fc_kHz",
         "B_kHz", "Tp_us", "TBP",
         "f_low_Hz", "f_high_Hz",
         "hardware_valid"]
    ]
)

,waveform_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,f_low_Hz,f_high_Hz,hardware_valid
0,W01,LFM,75,1.673603,1000,1.673603,74163.198349,75836.801651,True
1,W02,LFM,100,1.673603,1000,1.673603,99163.198349,100836.801651,True
2,W03,LFM,150,1.673603,1000,1.673603,149163.198349,150836.801651,True
3,W04,LFM,200,1.673603,1000,1.673603,199163.198349,200836.801651,True
4,W05,LFM,300,1.673603,1000,1.673603,299163.198349,300836.801651,True
5,W06,LFM,400,1.673603,1000,1.673603,399163.198349,400836.801651,True
6,W07,LFM,500,1.673603,1000,1.673603,499163.198349,500836.801651,True
7,W08,LFM,75,13.388826,2000,26.777653,68305.586789,81694.413211,True
8,W09,Geometric_NFM,75,1.673603,1000,1.673603,74163.198349,75836.801651,True
9,W10,Geometric_NFM,100,1.673603,1000,1.673603,99163.198349,100836.801651,True


In [ ]:
# ============================================================
# FINAL LUT SELECTION — STEP 1
# Balanced scoring of the 24-waveform library
# ============================================================

lut_df = evaluation_24_df.copy()

# Keep only hard-feasible candidates
lut_df = lut_df[lut_df["hard_feasible"]].copy()

# ------------------------------------------------------------
# 1. Energy score
# Lower energy is better.
# Log scale prevents large energy values from dominating.
# ------------------------------------------------------------
lut_df["log_energy"] = np.log10(
    lut_df["energy_proxy"].clip(lower=1e-12)
)

# Normalize within each operating state
energy_min = lut_df.groupby("operating_state_id")["log_energy"].transform("min")
energy_max = lut_df.groupby("operating_state_id")["log_energy"].transform("max")

lut_df["energy_score"] = np.where(
    energy_max > energy_min,
    1 - (lut_df["log_energy"] - energy_min) /
        (energy_max - energy_min),
    1.0
)

# ------------------------------------------------------------
# 2. Detection-margin score
# Do not reward enormous margins indefinitely.
# Cap useful margin at 20 dB.
# ------------------------------------------------------------
margin_capped = lut_df["detection_margin_dB"].clip(
    lower=0,
    upper=20
)

lut_df["margin_score"] = margin_capped / 20.0

# ------------------------------------------------------------
# 3. Resolution score
# Required resolution = 1 m.
# Reward better resolution, but only moderately.
# ------------------------------------------------------------
lut_df["resolution_score"] = (
    1.0 - lut_df["range_resolution_m"] /
    REQUIRED_RANGE_RESOLUTION_M
).clip(lower=0, upper=1)

# ------------------------------------------------------------
# 4. Balanced final score
#
# Energy is primary because this is an energy-aware LUT.
# Detection robustness is second.
# Resolution is third.
# ------------------------------------------------------------
lut_df["final_score"] = (
    0.50 * lut_df["energy_score"] +
    0.30 * lut_df["margin_score"] +
    0.20 * lut_df["resolution_score"]
)

# Higher score = better candidate
lut_df = lut_df.sort_values(
    ["operating_state_id", "final_score"],
    ascending=[True, False]
)

# Best candidate for every operating state
best_lut_df = (
    lut_df
    .groupby("operating_state_id", as_index=False)
    .first()
)

print("Operating states with feasible solution:",
      best_lut_df["operating_state_id"].nunique())

print("Total operating states:",
      operating_states_df["operating_state_id"].nunique())

print("\nSelected waveform count:")
print(best_lut_df["waveform_id"].nunique())

print("\nWaveform usage:")
print(
    best_lut_df["waveform_id"]
    .value_counts()
    .sort_index()
)

Operating states with feasible solution: 260
Total operating states: 390

Selected waveform count:
1

Waveform usage:
waveform_id
W08    260
Name: count, dtype: int64


In [ ]:
# ============================================================
# FINAL LUT SELECTION — STEP 2
# Scenario-wise selected waveform distribution
# ============================================================

print("Selected waveforms by scenario:\n")

scenario_waveform_usage = pd.crosstab(
    best_lut_df["scenario_id"],
    best_lut_df["waveform_id"]
)

display(scenario_waveform_usage)

print("\nSelected waveform families:\n")

family_usage = pd.crosstab(
    best_lut_df["scenario_id"],
    best_lut_df["waveform_type"]
)

display(family_usage)

print("\nFinal LUT summary:\n")

display(
    best_lut_df[
        [
            "operating_state_id",
            "environment_state_id",
            "scenario_id",
            "waveform_id",
            "waveform_type",
            "fc_kHz",
            "B_kHz",
            "Tp_us",
            "TBP",
            "range_resolution_m",
            "SE_proxy_dB",
            "detection_margin_dB",
            "energy_proxy",
            "final_score"
        ]
    ].head(20)
)

Selected waveforms by scenario:



waveform_id,W08
scenario_id,
S1_Near_LowNoise,130
S2_Medium,130



Selected waveform families:



waveform_type,LFM
scenario_id,
S1_Near_LowNoise,130
S2_Medium,130



Final LUT summary:



,operating_state_id,environment_state_id,scenario_id,waveform_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,range_resolution_m,SE_proxy_dB,detection_margin_dB,energy_proxy,final_score
0,OP_001,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S1_Near_LowNoise,W08,LFM,75,13.388826,2000,26.777653,0.055627,64.263333,54.263333,0.007494,0.988875
1,OP_002,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S2_Medium,W08,LFM,75,13.388826,2000,26.777653,0.055627,11.246963,1.246963,1500.837452,0.707579
2,OP_004,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S1_Near_LowNoise,W08,LFM,75,13.388826,2000,26.777653,0.055960,64.263333,54.263333,0.007494,0.988808
3,OP_005,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S2_Medium,W08,LFM,75,13.388826,2000,26.777653,0.055960,11.246963,1.246963,1500.837452,0.707512
4,OP_007,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S1_Near_LowNoise,W08,LFM,75,13.388826,2000,26.777653,0.055894,64.263333,54.263333,0.007494,0.988821
5,OP_008,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S2_Medium,W08,LFM,75,13.388826,2000,26.777653,0.055894,11.246963,1.246963,1500.837452,0.707526
6,OP_010,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1,S1_Near_LowNoise,W08,LFM,75,13.388826,2000,26.777653,0.056045,64.263333,54.263333,0.007494,0.988791
7,OP_011,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1,S2_Medium,W08,LFM,75,13.388826,2000,26.777653,0.056045,11.246963,1.246963,1500.837452,0.707496
8,OP_013,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q3,S1_Near_LowNoise,W08,LFM,75,13.388826,2000,26.777653,0.055776,64.263333,54.263333,0.007494,0.988845
9,OP_014,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q3,S2_Medium,W08,LFM,75,13.388826,2000,26.777653,0.055776,11.246963,1.246963,1500.837452,0.707549


In [ ]:
# ============================================================
# FINAL LUT SELECTION — ENERGY-AWARE + DETECTION CONSTRAINT
# ============================================================

lut_candidates = evaluation_24_df.copy()

# ------------------------------------------------------------
# Hard feasibility
# ------------------------------------------------------------
lut_candidates = lut_candidates[
    lut_candidates["hard_feasible"]
].copy()

# ------------------------------------------------------------
# Detection margin requirement for optimization
#
# We already require SE >= 10 dB.
# Here we use an additional robustness target of 5 dB.
# This prevents selecting a barely-feasible candidate
# when a significantly lower-energy robust candidate exists.
# ------------------------------------------------------------

ROBUST_MARGIN_DB = 5.0

lut_candidates["robust_detection"] = (
    lut_candidates["detection_margin_dB"] >= ROBUST_MARGIN_DB
)

# ------------------------------------------------------------
# Energy
#
# Primary objective: minimum pulse-energy proxy.
# Use logarithm because the energy range is highly skewed.
# ------------------------------------------------------------

lut_candidates["log_energy"] = np.log10(
    lut_candidates["energy_proxy"].clip(lower=1e-12)
)

# ------------------------------------------------------------
# Resolution
#
# All candidates are already <= 1 m.
# We only use resolution as a secondary preference.
# Target = 0.5 m.
# ------------------------------------------------------------

lut_candidates["resolution_excess"] = (
    lut_candidates["range_resolution_m"]
    / REQUIRED_RANGE_RESOLUTION_M
)

# ------------------------------------------------------------
# Detection margin excess
#
# Once 5 dB margin is achieved, additional margin has
# diminishing value.
# ------------------------------------------------------------

lut_candidates["margin_excess_dB"] = (
    lut_candidates["detection_margin_dB"]
    .clip(lower=0, upper=15)
)

# ------------------------------------------------------------
# Rank candidates WITHIN EACH OPERATING STATE
#
# Priority:
#   1. Robust detection if available
#   2. Lowest energy
#   3. Better resolution
#   4. Higher detection margin
# ------------------------------------------------------------

lut_candidates["robust_priority"] = (
    lut_candidates["robust_detection"].astype(int)
)

lut_candidates = lut_candidates.sort_values(
    [
        "operating_state_id",
        "robust_priority",
        "energy_proxy",
        "range_resolution_m",
        "detection_margin_dB"
    ],
    ascending=[
        True,
        False,
        True,
        True,
        False
    ]
)

# Select first candidate per operating state
final_lut_df = (
    lut_candidates
    .groupby("operating_state_id", as_index=False)
    .first()
)

# ------------------------------------------------------------
# Add status for states with NO feasible waveform
# ------------------------------------------------------------

all_states = operating_states_df[
    [
        "operating_state_id",
        "environment_state_id",
        "scenario_id",
        "range_m",
        "ambient_noise_dB",
        "target_strength_dB"
    ]
].copy()

final_lut_df = all_states.merge(
    final_lut_df,
    on=[
        "operating_state_id",
        "environment_state_id",
        "scenario_id",
        "range_m",
        "ambient_noise_dB",
        "target_strength_dB"
    ],
    how="left"
)

final_lut_df["lut_status"] = np.where(
    final_lut_df["waveform_id"].notna(),
    "FEASIBLE",
    "NO_FEASIBLE_WAVEFORM"
)

print("================================================")
print("FINAL LUT STATUS")
print("================================================")

print(
    final_lut_df["lut_status"]
    .value_counts()
)

print("\nWaveforms selected:")
print(
    final_lut_df.loc[
        final_lut_df["lut_status"] == "FEASIBLE",
        "waveform_id"
    ].value_counts().sort_index()
)

print("\nScenario status:")
print(
    pd.crosstab(
        final_lut_df["scenario_id"],
        final_lut_df["lut_status"]
    )
)

FINAL LUT STATUS
lut_status
FEASIBLE                260
NO_FEASIBLE_WAVEFORM    130
Name: count, dtype: int64

Waveforms selected:
waveform_id
W08    260
Name: count, dtype: int64

Scenario status:
lut_status        FEASIBLE  NO_FEASIBLE_WAVEFORM
scenario_id                                     
S1_Near_LowNoise       130                     0
S2_Medium              130                     0
S3_Far_HighNoise         0                   130


In [ ]:
# ============================================================
# FINAL LUT SUMMARY
# ============================================================

display(
    final_lut_df[
        [
            "operating_state_id",
            "environment_state_id",
            "scenario_id",
            "waveform_id",
            "waveform_type",
            "fc_kHz",
            "B_kHz",
            "Tp_us",
            "TBP",
            "range_resolution_m",
            "SE_proxy_dB",
            "detection_margin_dB",
            "energy_proxy",
            "lut_status"
        ]
    ].head(30)
)

print("\nSelected waveform count:",
      final_lut_df["waveform_id"].nunique())

print("\nSelected waveform IDs:")
print(
    sorted(
        final_lut_df["waveform_id"]
        .dropna()
        .unique()
    )
)

,operating_state_id,environment_state_id,scenario_id,waveform_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,range_resolution_m,SE_proxy_dB,detection_margin_dB,energy_proxy,lut_status
0,OP_001,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S1_Near_LowNoise,W08,LFM,75.0,13.388826,2000.0,26.777653,0.055627,64.263333,54.263333,0.007494,FEASIBLE
1,OP_002,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S2_Medium,W08,LFM,75.0,13.388826,2000.0,26.777653,0.055627,11.246963,1.246963,1500.837452,FEASIBLE
2,OP_003,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S3_Far_HighNoise,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_FEASIBLE_WAVEFORM
3,OP_004,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S1_Near_LowNoise,W08,LFM,75.0,13.388826,2000.0,26.777653,0.055960,64.263333,54.263333,0.007494,FEASIBLE
4,OP_005,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S2_Medium,W08,LFM,75.0,13.388826,2000.0,26.777653,0.055960,11.246963,1.246963,1500.837452,FEASIBLE
5,OP_006,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S3_Far_HighNoise,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_FEASIBLE_WAVEFORM
6,OP_007,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S1_Near_LowNoise,W08,LFM,75.0,13.388826,2000.0,26.777653,0.055894,64.263333,54.263333,0.007494,FEASIBLE
7,OP_008,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S2_Medium,W08,LFM,75.0,13.388826,2000.0,26.777653,0.055894,11.246963,1.246963,1500.837452,FEASIBLE
8,OP_009,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S3_Far_HighNoise,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_FEASIBLE_WAVEFORM
9,OP_010,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1,S1_Near_LowNoise,W08,LFM,75.0,13.388826,2000.0,26.777653,0.056045,64.263333,54.263333,0.007494,FEASIBLE



Selected waveform count: 1

Selected waveform IDs:
['W08']


In [ ]:
# ============================================================
# S2 FEASIBILITY BREAKDOWN
# ============================================================

s2_feasible = evaluation_24_df[
    (evaluation_24_df["scenario_id"] == "S2_Medium") &
    (evaluation_24_df["hard_feasible"])
].copy()

print("Total S2 feasible evaluations:", len(s2_feasible))

print("\nS2 feasible waveform IDs:")
print(
    s2_feasible["waveform_id"]
    .value_counts()
    .sort_index()
)

print("\nS2 feasible waveform details:")
display(
    s2_feasible[
        [
            "waveform_id",
            "waveform_type",
            "fc_kHz",
            "B_kHz",
            "Tp_us",
            "TBP",
            "range_resolution_m",
            "SE_proxy_dB",
            "detection_margin_dB",
            "energy_proxy"
        ]
    ]
    .drop_duplicates("waveform_id")
    .sort_values(["Tp_us", "fc_kHz"])
)

Total S2 feasible evaluations: 260

S2 feasible waveform IDs:
waveform_id
W08    130
W16    130
Name: count, dtype: int64

S2 feasible waveform details:


,waveform_id,waveform_type,fc_kHz,B_kHz,Tp_us,TBP,range_resolution_m,SE_proxy_dB,detection_margin_dB,energy_proxy
31,W08,LFM,75,13.388826,2000,26.777653,0.055627,11.246963,1.246963,1500.837452
39,W16,Geometric_NFM,75,13.388826,2000,26.777653,0.055627,11.246963,1.246963,1500.837452


In [ ]:
# ============================================================
# OJAS FINAL LUT
# 24-waveform library + operating-state mapping
# ============================================================

# ------------------------------------------------------------
# Final policy under the validated/current model:
#
# S1 + S2:
#   W08 = primary selected waveform
#
# S3:
#   NO_FEASIBLE_WAVEFORM
#
# W16:
#   retained as Geometric-NFM alternative to W08
# ------------------------------------------------------------

final_lut = final_lut_df.copy()

# Explicitly enforce the final selected waveform for
# every currently feasible operating state.
final_lut.loc[
    final_lut["lut_status"] == "FEASIBLE",
    "waveform_id"
] = "W08"

# Add explicit selection role
final_lut["selection_role"] = np.where(
    final_lut["lut_status"] == "FEASIBLE",
    "PRIMARY",
    "NO_FEASIBLE_SOLUTION"
)

# ------------------------------------------------------------
# Keep only useful deployment columns
# ------------------------------------------------------------

final_lut = final_lut[
    [
        "operating_state_id",
        "environment_state_id",
        "scenario_id",

        "depth_m",
        "temperature_C",
        "salinity_PSU",
        "turbidity_NTU",
        "c_mackenzie_mps",

        "range_m",
        "ambient_noise_dB",
        "target_strength_dB",

        "waveform_id",
        "waveform_type",
        "fc_kHz",
        "B_kHz",
        "Tp_us",
        "TBP",

        "range_resolution_m",
        "SE_proxy_dB",
        "detection_margin_dB",
        "energy_proxy",

        "lut_status",
        "selection_role"
    ]
].copy()

# ------------------------------------------------------------
# Fill waveform metadata for W08 explicitly
# ------------------------------------------------------------

w08 = final_waveform_df[
    final_waveform_df["waveform_id"] == "W08"
].iloc[0]

feasible_mask = final_lut["lut_status"] == "FEASIBLE"

final_lut.loc[feasible_mask, "waveform_type"] = w08["waveform_type"]
final_lut.loc[feasible_mask, "fc_kHz"] = w08["fc_kHz"]
final_lut.loc[feasible_mask, "B_kHz"] = w08["B_kHz"]
final_lut.loc[feasible_mask, "Tp_us"] = w08["Tp_us"]
final_lut.loc[feasible_mask, "TBP"] = w08["TBP"]

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 65)
print("OJAS FINAL LUT")
print("=" * 65)

print("\nTotal operating states:",
      len(final_lut))

print("\nLUT status:")
print(final_lut["lut_status"].value_counts())

print("\nSelected waveform:")
print(
    final_lut.loc[
        feasible_mask, "waveform_id"
    ].value_counts()
)

print("\nScenario-wise result:")
print(
    pd.crosstab(
        final_lut["scenario_id"],
        final_lut["lut_status"]
    )
)

print("\nFinal LUT preview:")
display(final_lut.head(15))

OJAS FINAL LUT

Total operating states: 390

LUT status:
lut_status
FEASIBLE                260
NO_FEASIBLE_WAVEFORM    130
Name: count, dtype: int64

Selected waveform:
waveform_id
W08    260
Name: count, dtype: int64

Scenario-wise result:
lut_status        FEASIBLE  NO_FEASIBLE_WAVEFORM
scenario_id                                     
S1_Near_LowNoise       130                     0
S2_Medium              130                     0
S3_Far_HighNoise         0                   130

Final LUT preview:


,operating_state_id,environment_state_id,scenario_id,depth_m,temperature_C,salinity_PSU,turbidity_NTU,c_mackenzie_mps,range_m,ambient_noise_dB,...,fc_kHz,B_kHz,Tp_us,TBP,range_resolution_m,SE_proxy_dB,detection_margin_dB,energy_proxy,lut_status,selection_role
0,OP_001,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S1_Near_LowNoise,3.0,11.676825,29.907525,0.761975,1489.552899,100,50,...,75.0,13.388826,2000.0,26.777653,0.055627,64.263333,54.263333,0.007494,FEASIBLE,PRIMARY
1,OP_002,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S2_Medium,3.0,11.676825,29.907525,0.761975,1489.552899,500,65,...,75.0,13.388826,2000.0,26.777653,0.055627,11.246963,1.246963,1500.837452,FEASIBLE,PRIMARY
2,OP_003,DEPTH_Q1_TEMP_Q1_SAL_Q1_TURB_Q3,S3_Far_HighNoise,NaN,NaN,NaN,NaN,NaN,1000,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_FEASIBLE_WAVEFORM,NO_FEASIBLE_SOLUTION
3,OP_004,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S1_Near_LowNoise,5.0,12.451406,35.029951,0.727600,1498.474219,100,50,...,75.0,13.388826,2000.0,26.777653,0.055960,64.263333,54.263333,0.007494,FEASIBLE,PRIMARY
4,OP_005,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S2_Medium,5.0,12.451406,35.029951,0.727600,1498.474219,500,65,...,75.0,13.388826,2000.0,26.777653,0.055960,11.246963,1.246963,1500.837452,FEASIBLE,PRIMARY
5,OP_006,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q2,S3_Far_HighNoise,NaN,NaN,NaN,NaN,NaN,1000,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_FEASIBLE_WAVEFORM,NO_FEASIBLE_SOLUTION
6,OP_007,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S1_Near_LowNoise,3.5,11.847276,35.303154,0.745325,1496.719121,100,50,...,75.0,13.388826,2000.0,26.777653,0.055894,64.263333,54.263333,0.007494,FEASIBLE,PRIMARY
7,OP_008,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S2_Medium,3.5,11.847276,35.303154,0.745325,1496.719121,500,65,...,75.0,13.388826,2000.0,26.777653,0.055894,11.246963,1.246963,1500.837452,FEASIBLE,PRIMARY
8,OP_009,DEPTH_Q1_TEMP_Q1_SAL_Q2_TURB_Q3,S3_Far_HighNoise,NaN,NaN,NaN,NaN,NaN,1000,80,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO_FEASIBLE_WAVEFORM,NO_FEASIBLE_SOLUTION
9,OP_010,DEPTH_Q1_TEMP_Q2_SAL_Q1_TURB_Q1,S1_Near_LowNoise,1.0,16.602250,25.057100,0.404350,1500.745861,100,50,...,75.0,13.388826,2000.0,26.777653,0.056045,64.263333,54.263333,0.007494,FEASIBLE,PRIMARY


In [ ]:
# ============================================================
# SAVE DEPLOYABLE WAVEFORM LIBRARY
# ============================================================

waveform_library_csv = "OJAS_final_waveform_library_24.csv"

final_waveform_df.to_csv(
    waveform_library_csv,
    index=False
)

print(f"Saved: {waveform_library_csv}")

Saved: OJAS_final_waveform_library_24.csv
